<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/14_Statistical_Fidelity_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
# =============================================================================
# NOTEBOOK 14 — STATISTICAL FIDELITY EVALUATION
# SPP-GAN Research Project | Section 1: Header & Scope
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import warnings
import platform

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# -----------------------------------------------------------------------------
# 1. CANONICAL PROJECT PATHS
# -----------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")
NB02_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_02"
NB13_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_13"
NB14_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_14"

NB14_CONFIG_ROOT = NB14_ROOT / "configuration"
NB14_INPUT_ROOT = NB14_ROOT / "input_audit"
NB14_FEATURE_ROOT = NB14_ROOT / "feature_results"
NB14_DATASET_ROOT = NB14_ROOT / "dataset_results"
NB14_AGG_ROOT = NB14_ROOT / "aggregations"
NB14_VALIDATION_ROOT = NB14_ROOT / "validation"
NB14_LOG_ROOT = NB14_ROOT / "logs"

for folder in [
    NB14_CONFIG_ROOT,
    NB14_INPUT_ROOT,
    NB14_FEATURE_ROOT,
    NB14_DATASET_ROOT,
    NB14_AGG_ROOT,
    NB14_VALIDATION_ROOT,
    NB14_LOG_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. DATASET REGISTRY
# -----------------------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

ID_COLUMNS = [
    "__original_row_id__",
    "encounter_id",
    "patient_nbr",
]

SYNTHETIC_FILES = {
    dataset: (
        NB13_ROOT
        / "synthetic_datasets"
        / f"{dataset}_sppgan_synthetic.csv"
    )
    for dataset in DATASET_IDS
}

# Notebook 02 candidate locations for held-out real test data.
REAL_TEST_CANDIDATES = {
    dataset: [
        NB02_ROOT / dataset / "test.csv",
        NB02_ROOT / "splits" / dataset / "test.csv",
        NB02_ROOT / "test" / f"{dataset}_test.csv",
        NB02_ROOT / f"{dataset}_test.csv",
    ]
    for dataset in DATASET_IDS
}

# -----------------------------------------------------------------------------
# 3. REPRODUCIBILITY AND EVALUATION CONFIGURATION
# -----------------------------------------------------------------------------

MASTER_SEED = 2025
RANDOM_STATE = MASTER_SEED

# Distribution representations
QUANTILE_GRID_SIZE = 101
HISTOGRAM_BINS = 50
PROBABILITY_SMOOTHING = 1e-8
MAX_CATEGORY_LEVELS = 5000

# Resource controls
READ_CSV_CHUNKSIZE = 100_000
MAX_CORRELATION_FEATURES = 100
MIN_VALID_NUMERIC_ROWS = 2

# Methodology: test data are evaluation-only; no fitting or tuning on test data.
REAL_REFERENCE_SPLIT = "test"
STATISTICAL_PROFILE_REFERENCE = "training"
GENERATION_SOURCE = "Notebook 13 canonical synthetic CSV"

NOTEBOOK_NAME = "Notebook 14 — Statistical Fidelity Evaluation"
NOTEBOOK_VERSION = "1.1.0"

warnings.filterwarnings("ignore", category=RuntimeWarning)

# -----------------------------------------------------------------------------
# 4. PATH VALIDATION
# -----------------------------------------------------------------------------

required_directories = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "NB02_ROOT": NB02_ROOT,
    "NB13_ROOT": NB13_ROOT,
}

for name, path in required_directories.items():
    if not path.exists() or not path.is_dir():
        raise FileNotFoundError(
            f"{name} does not exist or is not a directory: {path}"
        )

# -----------------------------------------------------------------------------
# 5. CONFIGURATION PERSISTENCE
# -----------------------------------------------------------------------------

NOTEBOOK14_CONFIG = {
    "notebook_name": NOTEBOOK_NAME,
    "notebook_version": NOTEBOOK_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook_02_root": str(NB02_ROOT),
    "notebook_13_root": str(NB13_ROOT),
    "notebook_14_root": str(NB14_ROOT),
    "datasets": DATASET_IDS,
    "target_columns": TARGET_COLUMNS,
    "excluded_identifier_columns": ID_COLUMNS,
    "real_reference_split": REAL_REFERENCE_SPLIT,
    "statistical_profile_reference": STATISTICAL_PROFILE_REFERENCE,
    "generation_source": GENERATION_SOURCE,
    "master_seed": MASTER_SEED,
    "quantile_grid_size": QUANTILE_GRID_SIZE,
    "histogram_bins": HISTOGRAM_BINS,
    "probability_smoothing": PROBABILITY_SMOOTHING,
    "max_category_levels": MAX_CATEGORY_LEVELS,
    "max_correlation_features": MAX_CORRELATION_FEATURES,
    "minimum_valid_numeric_rows": MIN_VALID_NUMERIC_ROWS,
    "environment": {
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
    },
}

CONFIG_PATH = (
    NB14_CONFIG_ROOT / "statistical_fidelity_configuration.json"
)

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(NOTEBOOK14_CONFIG, file, indent=2, ensure_ascii=False)

# Verify that the configuration was saved correctly.
with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    CONFIG_CHECK = json.load(file)

assert CONFIG_CHECK["datasets"] == DATASET_IDS
assert CONFIG_CHECK["target_columns"] == TARGET_COLUMNS

# -----------------------------------------------------------------------------
# 6. INITIALIZATION SUMMARY
# -----------------------------------------------------------------------------

print("=" * 100)
print(NOTEBOOK_NAME)
print("=" * 100)
print("Project root       :", PROJECT_ROOT)
print("Notebook 02 root   :", NB02_ROOT)
print("Notebook 13 root   :", NB13_ROOT)
print("Notebook 14 root   :", NB14_ROOT)
print("Datasets           :", DATASET_IDS)
print("Target columns     :", TARGET_COLUMNS)
print("Real reference     :", REAL_REFERENCE_SPLIT)
print("Master seed        :", MASTER_SEED)
print("Notebook version   :", NOTEBOOK_VERSION)
print("Configuration saved:", CONFIG_PATH)
print("Configuration check: PASS")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Notebook 14 — Statistical Fidelity Evaluation
Project root       : /content/drive/MyDrive/SPP_GAN_Research
Notebook 02 root   : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
Notebook 13 root   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13
Notebook 14 root   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14
Datasets           : ['adult_income', 'bank_marketing', 'diabetes_130us']
Target columns     : {'adult_income': 'income', 'bank_marketing': 'y', 'diabetes_130us': 'readmitted'}
Real reference     : test
Master seed        : 2025
Notebook version   : 1.1.0
Configuration saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/configuration/statistical_fidelity_configuration.json
Configuration check: PASS


In [29]:
# =============================================================================
# SECTION 2 — LOAD REAL TEST DATA
# =============================================================================

import json
import hashlib
from pathlib import Path

# Canonical Notebook 02 native split directory.
NB02_NATIVE_ROOT = NB02_ROOT / "native"

# Notebook 02 canonical native manifest.
NB02_NATIVE_MANIFEST = (
    NB02_NATIVE_ROOT / "native_dataset_manifest.csv"
)

# Combined split provenance manifest created by Notebook 02.
NB02_SPLIT_MANIFEST = (
    PROJECT_ROOT
    / "results"
    / "raw_validation"
    / "notebook_02_manifests"
    / "combined_split_manifest.csv"
)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_real_test_path(dataset_id):
    """
    Resolve the canonical Notebook 02 native test CSV.
    Do not substitute training or validation data.
    """
    path = NB02_NATIVE_ROOT / dataset_id / "test.csv"

    if not path.is_file():
        raise FileNotFoundError(
            f"Canonical real test CSV not found for {dataset_id}: {path}"
        )

    return path


# Validate canonical manifests before loading the test data.
if not NB02_NATIVE_MANIFEST.is_file():
    raise FileNotFoundError(
        f"Notebook 02 native manifest missing: {NB02_NATIVE_MANIFEST}"
    )

if not NB02_SPLIT_MANIFEST.is_file():
    raise FileNotFoundError(
        f"Notebook 02 split provenance manifest missing: {NB02_SPLIT_MANIFEST}"
    )

native_manifest_df = pd.read_csv(NB02_NATIVE_MANIFEST, low_memory=False)
split_manifest_df = pd.read_csv(NB02_SPLIT_MANIFEST, low_memory=False)

required_split_columns = {
    "dataset_id", "split", "__original_row_id__"
}
missing_manifest_columns = (
    required_split_columns - set(split_manifest_df.columns)
)

if missing_manifest_columns:
    raise ValueError(
        "Notebook 02 split manifest is missing required columns: "
        f"{sorted(missing_manifest_columns)}"
    )

REAL_TEST_DATA = {}
REAL_TEST_PATHS = {}
REAL_TEST_FINGERPRINTS = {}
REAL_TEST_MANIFEST_ROWS = []

for dataset_id in DATASET_IDS:
    path = resolve_real_test_path(dataset_id)
    target = TARGET_COLUMNS[dataset_id]

    df = pd.read_csv(path, low_memory=False)

    if df.empty:
        raise ValueError(f"Real test dataset is empty: {path}")

    if target not in df.columns:
        raise ValueError(
            f"Target '{target}' is missing from {dataset_id} test data."
        )

    if not df.columns.is_unique:
        raise ValueError(
            f"Duplicate column names found in real test data: {dataset_id}"
        )

    # Verify that manifest provenance records agree with the test CSV.
    test_manifest = split_manifest_df[
        (split_manifest_df["dataset_id"] == dataset_id)
        & (split_manifest_df["split"].astype(str).str.lower() == "test")
    ].copy()

    if test_manifest.empty:
        raise ValueError(
            f"No TEST records found in Notebook 02 split manifest "
            f"for {dataset_id}."
        )

    if "__original_row_id__" not in df.columns:
        raise ValueError(
            f"Provenance column '__original_row_id__' missing from "
            f"{dataset_id} test CSV; cannot verify split membership."
        )

    csv_ids = set(df["__original_row_id__"].astype(str))
    manifest_ids = set(
        test_manifest["__original_row_id__"].astype(str)
    )

    if csv_ids != manifest_ids:
        raise ValueError(
            f"Test CSV provenance does not match the Notebook 02 "
            f"split manifest for {dataset_id}. "
            f"CSV IDs={len(csv_ids)}, manifest IDs={len(manifest_ids)}."
        )

    # Preserve provenance for audit here; Section 4 excludes it from metrics.
    REAL_TEST_DATA[dataset_id] = df
    REAL_TEST_PATHS[dataset_id] = path
    REAL_TEST_FINGERPRINTS[dataset_id] = sha256_file(path)

    REAL_TEST_MANIFEST_ROWS.append({
        "dataset_id": dataset_id,
        "split": "test",
        "path": str(path),
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "target_column": target,
        "provenance_count": int(df["__original_row_id__"].nunique()),
        "manifest_provenance_count": int(len(test_manifest)),
        "provenance_match": True,
        "sha256": REAL_TEST_FINGERPRINTS[dataset_id],
        "status": "PASS",
    })

    print(
        f"✓ {dataset_id:18s} | shape={df.shape} | "
        f"target={target} | split=TEST | "
        f"provenance=PASS | file={path.name}"
    )

REAL_TEST_MANIFEST_VALIDATION_DF = pd.DataFrame(
    REAL_TEST_MANIFEST_ROWS
)

REAL_TEST_MANIFEST_VALIDATION_PATH = (
    NB14_VALIDATION_ROOT / "real_test_split_validation.csv"
)
REAL_TEST_MANIFEST_VALIDATION_DF.to_csv(
    REAL_TEST_MANIFEST_VALIDATION_PATH,
    index=False
)

print("\nReal test datasets loaded:", len(REAL_TEST_DATA))
print("Notebook 02 native manifest:", NB02_NATIVE_MANIFEST)
print("Split provenance manifest:", NB02_SPLIT_MANIFEST)
print("Validation report:", REAL_TEST_MANIFEST_VALIDATION_PATH)

assert len(REAL_TEST_DATA) == len(DATASET_IDS)
assert REAL_TEST_MANIFEST_VALIDATION_DF["status"].eq("PASS").all()

✓ adult_income       | shape=(7327, 16) | target=income | split=TEST | provenance=PASS | file=test.csv
✓ bank_marketing     | shape=(6782, 18) | target=y | split=TEST | provenance=PASS | file=test.csv
✓ diabetes_130us     | shape=(15265, 49) | target=readmitted | split=TEST | provenance=PASS | file=test.csv

Real test datasets loaded: 3
Notebook 02 native manifest: /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
Split provenance manifest: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/combined_split_manifest.csv
Validation report: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/real_test_split_validation.csv


In [30]:
# =============================================================================
# SECTION 3 — LOAD SYNTHETIC DATA
# =============================================================================

SYNTHETIC_DATA = {}
SYNTHETIC_PATHS = {}

for dataset_id in DATASET_IDS:
    path = SYNTHETIC_FILES[dataset_id]

    if not path.exists():
        raise FileNotFoundError(
            f"Notebook 13 synthetic CSV is missing: {path}"
        )

    df = pd.read_csv(path, low_memory=False)

    if df.empty:
        raise ValueError(f"Synthetic dataset is empty: {path}")

    target = TARGET_COLUMNS[dataset_id]
    if target not in df.columns:
        raise ValueError(
            f"Synthetic target '{target}' missing for {dataset_id}. "
            "Use the final Notebook 13 CSV containing the assigned target."
        )

    SYNTHETIC_DATA[dataset_id] = df
    SYNTHETIC_PATHS[dataset_id] = path

    print(
        f"✓ {dataset_id:18s} | shape={df.shape} | "
        f"target={target} | file={path.name}"
    )

print("\nSynthetic datasets loaded:", len(SYNTHETIC_DATA))

✓ adult_income       | shape=(34189, 15) | target=income | file=adult_income_sppgan_synthetic.csv
✓ bank_marketing     | shape=(31647, 17) | target=y | file=bank_marketing_sppgan_synthetic.csv
✓ diabetes_130us     | shape=(71236, 48) | target=readmitted | file=diabetes_130us_sppgan_synthetic.csv

Synthetic datasets loaded: 3


In [32]:
# =============================================================================
# SECTION 4 — VALIDATE SCHEMAS
# SPP-GAN | Held-out test reference and synthetic schema validation
# =============================================================================

SCHEMA_VALIDATION_ROWS = []
REAL_EVAL_DATA = {}
SYNTHETIC_EVAL_DATA = {}

for dataset_id in DATASET_IDS:
    real_df = REAL_TEST_DATA[dataset_id].copy()
    synth_df = SYNTHETIC_DATA[dataset_id].copy()

    target = TARGET_COLUMNS[dataset_id]

    # -------------------------------------------------------------------------
    # 1. Validate unique column names
    # -------------------------------------------------------------------------
    if not real_df.columns.is_unique:
        raise ValueError(
            f"Duplicate column names in real test data: {dataset_id}"
        )

    if not synth_df.columns.is_unique:
        raise ValueError(
            f"Duplicate column names in synthetic data: {dataset_id}"
        )

    # -------------------------------------------------------------------------
    # 2. Exclude provenance and identifier fields from fidelity evaluation
    # -------------------------------------------------------------------------
    excluded_real = [
        col for col in ID_COLUMNS if col in real_df.columns
    ]
    excluded_synthetic = [
        col for col in ID_COLUMNS if col in synth_df.columns
    ]

    real_df = real_df.drop(columns=excluded_real, errors="ignore")
    synth_df = synth_df.drop(columns=excluded_synthetic, errors="ignore")

    real_columns = list(real_df.columns)
    synth_columns = list(synth_df.columns)

    missing_in_synthetic = sorted(set(real_columns) - set(synth_columns))
    extra_in_synthetic = sorted(set(synth_columns) - set(real_columns))

    same_column_set = set(real_columns) == set(synth_columns)
    same_column_order = real_columns == synth_columns

    target_in_real = target in real_columns
    target_in_synthetic = target in synth_columns

    # -------------------------------------------------------------------------
    # 3. Validate schema and target integrity
    # -------------------------------------------------------------------------
    if not target_in_real or not target_in_synthetic:
        raise ValueError(
            f"Target '{target}' missing after identifier exclusion "
            f"for {dataset_id}."
        )

    if not same_column_set:
        raise ValueError(
            f"Schema mismatch for {dataset_id}.\n"
            f"Missing in synthetic: {missing_in_synthetic}\n"
            f"Extra in synthetic: {extra_in_synthetic}"
        )

    # Align column order; do not align or pair rows.
    synth_df = synth_df[real_columns].copy()
    real_df = real_df[real_columns].copy()

    # -------------------------------------------------------------------------
    # 4. Record schema audit
    # -------------------------------------------------------------------------
    SCHEMA_VALIDATION_ROWS.append({
        "dataset_id": dataset_id,
        "real_rows": int(len(real_df)),
        "synthetic_rows": int(len(synth_df)),
        "real_columns": int(len(real_columns)),
        "synthetic_columns": int(len(synth_df.columns)),
        "target_column": target,
        "target_in_real": bool(target_in_real),
        "target_in_synthetic": bool(target_in_synthetic),
        "excluded_real_identifiers": json.dumps(excluded_real),
        "excluded_synthetic_identifiers": json.dumps(excluded_synthetic),
        "missing_in_synthetic": json.dumps(missing_in_synthetic),
        "extra_in_synthetic": json.dumps(extra_in_synthetic),
        "same_column_set": bool(same_column_set),
        "same_column_order_before_alignment": bool(same_column_order),
        "same_column_order_after_alignment": bool(
            list(real_df.columns) == list(synth_df.columns)
        ),
        "schema_status": "PASS",
    })

    REAL_EVAL_DATA[dataset_id] = real_df
    SYNTHETIC_EVAL_DATA[dataset_id] = synth_df

SCHEMA_VALIDATION_DF = pd.DataFrame(SCHEMA_VALIDATION_ROWS)

SCHEMA_VALIDATION_PATH = (
    NB14_VALIDATION_ROOT / "schema_validation.csv"
)
SCHEMA_VALIDATION_DF.to_csv(
    SCHEMA_VALIDATION_PATH, index=False
)

display(SCHEMA_VALIDATION_DF)

print("✓ Schema validation passed for all datasets.")
print("✓ Target columns retained.")
print("✓ Identifier/provenance columns excluded from fidelity metrics.")
print("✓ Synthetic columns aligned to real test schema.")
print("✓ Real and synthetic rows remain unpaired.")
print("Saved:", SCHEMA_VALIDATION_PATH)

,dataset_id,real_rows,synthetic_rows,real_columns,synthetic_columns,target_column,target_in_real,target_in_synthetic,excluded_real_identifiers,excluded_synthetic_identifiers,missing_in_synthetic,extra_in_synthetic,same_column_set,same_column_order_before_alignment,same_column_order_after_alignment,schema_status
0,adult_income,7327,34189,15,15,income,True,True,"[""__original_row_id__""]",[],[],[],True,True,True,PASS
1,bank_marketing,6782,31647,17,17,y,True,True,"[""__original_row_id__""]",[],[],[],True,True,True,PASS
2,diabetes_130us,15265,71236,48,48,readmitted,True,True,"[""__original_row_id__""]",[],[],[],True,True,True,PASS


✓ Schema validation passed for all datasets.
✓ Target columns retained.
✓ Identifier/provenance columns excluded from fidelity metrics.
✓ Synthetic columns aligned to real test schema.
✓ Real and synthetic rows remain unpaired.
Saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/schema_validation.csv


In [34]:
# =============================================================================
# SECTION 5 — ALIGN REAL/SYNTHETIC FEATURES
# =============================================================================

NUMERIC_FEATURES = {}
CATEGORICAL_FEATURES = {}
FEATURE_TYPES = {}
FEATURE_ALIGNMENT_ROWS = []

for dataset_id in DATASET_IDS:
    real_df = REAL_EVAL_DATA[dataset_id]
    synth_df = SYNTHETIC_EVAL_DATA[dataset_id]
    target = TARGET_COLUMNS[dataset_id]

    if list(real_df.columns) != list(synth_df.columns):
        raise ValueError(
            f"{dataset_id}: real and synthetic schemas are not aligned."
        )

    if target not in real_df.columns:
        raise ValueError(
            f"{dataset_id}: target column '{target}' is missing."
        )

    numeric_cols = []
    categorical_cols = []

    for col in real_df.columns:
        real_series = real_df[col]
        synth_series = synth_df[col]

        # Keep the target categorical for distribution and classification-label
        # fidelity, regardless of its underlying dtype.
        if col == target:
            categorical_cols.append(col)
            continue

        real_is_numeric = pd.api.types.is_numeric_dtype(real_series)
        synth_is_numeric = pd.api.types.is_numeric_dtype(synth_series)

        # Classify a feature as numeric only when both datasets are numeric.
        # Do not silently coerce incompatible feature types.
        if real_is_numeric and synth_is_numeric:
            numeric_cols.append(col)
        else:
            categorical_cols.append(col)

        if real_is_numeric != synth_is_numeric:
            warnings.warn(
                f"{dataset_id}.{col}: dtype mismatch "
                f"(real={real_series.dtype}, synthetic={synth_series.dtype}); "
                "classified as categorical for this evaluation.",
                RuntimeWarning,
            )

    NUMERIC_FEATURES[dataset_id] = numeric_cols
    CATEGORICAL_FEATURES[dataset_id] = categorical_cols

    FEATURE_TYPES[dataset_id] = {
        **{col: "numeric" for col in numeric_cols},
        **{col: "categorical" for col in categorical_cols},
    }

    for col in real_df.columns:
        real_series = real_df[col]
        synth_series = synth_df[col]

        FEATURE_ALIGNMENT_ROWS.append({
            "dataset_id": dataset_id,
            "feature": col,
            "feature_type": FEATURE_TYPES[dataset_id][col],
            "is_target": col == target,
            "real_dtype": str(real_series.dtype),
            "synthetic_dtype": str(synth_series.dtype),
            "dtype_match": str(real_series.dtype) == str(synth_series.dtype),
            "real_missing_count": int(real_series.isna().sum()),
            "synthetic_missing_count": int(synth_series.isna().sum()),
            "real_missing_rate": float(real_series.isna().mean()),
            "synthetic_missing_rate": float(synth_series.isna().mean()),
            "real_unique_values": int(real_series.nunique(dropna=True)),
            "synthetic_unique_values": int(synth_series.nunique(dropna=True)),
        })

    print(
        f"✓ {dataset_id}: {len(numeric_cols)} numeric, "
        f"{len(categorical_cols)} categorical/target features"
    )

FEATURE_ALIGNMENT_DF = pd.DataFrame(FEATURE_ALIGNMENT_ROWS)

FEATURE_ALIGNMENT_PATH = NB14_VALIDATION_ROOT / "feature_alignment.csv"
FEATURE_ALIGNMENT_DF.to_csv(FEATURE_ALIGNMENT_PATH, index=False)

display(FEATURE_ALIGNMENT_DF.head(15))

print("\nFeature alignment summary:")
display(
    FEATURE_ALIGNMENT_DF.groupby(
        ["dataset_id", "feature_type"], dropna=False
    ).agg(
        feature_count=("feature", "count"),
        dtype_mismatches=("dtype_match", lambda x: int((~x).sum())),
        real_missing_values=("real_missing_count", "sum"),
        synthetic_missing_values=("synthetic_missing_count", "sum"),
    ).reset_index()
)

print("\n✓ Feature types established using aligned real and synthetic schemas.")
print("✓ Target columns retained as categorical features.")
print("✓ Missingness recorded without modifying the data.")
print("✓ Feature alignment saved:", FEATURE_ALIGNMENT_PATH)

✓ adult_income: 6 numeric, 9 categorical/target features
✓ bank_marketing: 7 numeric, 10 categorical/target features
✓ diabetes_130us: 11 numeric, 37 categorical/target features


,dataset_id,feature,feature_type,is_target,real_dtype,synthetic_dtype,dtype_match,real_missing_count,synthetic_missing_count,real_missing_rate,synthetic_missing_rate,real_unique_values,synthetic_unique_values
0,adult_income,age,numeric,False,int64,int64,True,0,0,0.000000,0.0,71,96
1,adult_income,workclass,categorical,False,object,object,True,379,0,0.051726,0.0,8,6
2,adult_income,fnlwgt,numeric,False,int64,int64,True,0,0,0.000000,0.0,6440,32666
3,adult_income,education,categorical,False,object,object,True,0,0,0.000000,0.0,16,16
4,adult_income,education_num,numeric,False,int64,int64,True,0,0,0.000000,0.0,16,20
5,adult_income,marital_status,categorical,False,object,object,True,0,0,0.000000,0.0,7,6
6,adult_income,occupation,categorical,False,object,object,True,382,0,0.052136,0.0,14,13
7,adult_income,relationship,categorical,False,object,object,True,0,0,0.000000,0.0,6,6
8,adult_income,race,categorical,False,object,object,True,0,0,0.000000,0.0,5,5
9,adult_income,sex,categorical,False,object,object,True,0,0,0.000000,0.0,2,2



Feature alignment summary:


,dataset_id,feature_type,feature_count,dtype_mismatches,real_missing_values,synthetic_missing_values
0,adult_income,categorical,9,0,891,0
1,adult_income,numeric,6,0,0,0
2,bank_marketing,categorical,10,0,0,0
3,bank_marketing,numeric,7,0,0,0
4,diabetes_130us,categorical,37,0,56123,0
5,diabetes_130us,numeric,11,0,0,0



✓ Feature types established using aligned real and synthetic schemas.
✓ Target columns retained as categorical features.
✓ Missingness recorded without modifying the data.
✓ Feature alignment saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/feature_alignment.csv


In [36]:
# =============================================================================
# SECTION 6 — NUMERICAL DISTRIBUTION EVALUATION
# =============================================================================

FEATURE_RESULT_MAP = {}
NUMERIC_QUANTILES = {}
CATEGORICAL_PROBABILITIES = {}
NUMERIC_HISTOGRAMS = {}

QUANTILE_LEVELS = np.linspace(0.0, 1.0, QUANTILE_GRID_SIZE)


def numeric_array(series):
    """Return finite numeric values without modifying the source data."""
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    return values[np.isfinite(values)]


def safe_numeric_summary(series):
    """Summarize valid numeric values while retaining missingness information."""
    values = numeric_array(series)

    if len(values) == 0:
        return {
            "valid_n": 0,
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "q25": np.nan,
            "median": np.nan,
            "q75": np.nan,
            "max": np.nan,
        }

    return {
        "valid_n": int(len(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
        "min": float(np.min(values)),
        "q25": float(np.quantile(values, 0.25)),
        "median": float(np.quantile(values, 0.50)),
        "q75": float(np.quantile(values, 0.75)),
        "max": float(np.max(values)),
    }


for dataset_id in DATASET_IDS:
    real_df = REAL_EVAL_DATA[dataset_id]
    synth_df = SYNTHETIC_EVAL_DATA[dataset_id]

    for feature in NUMERIC_FEATURES[dataset_id]:
        real_series = real_df[feature]
        synth_series = synth_df[feature]

        real_values = numeric_array(real_series)
        synth_values = numeric_array(synth_series)

        real_summary = safe_numeric_summary(real_series)
        synth_summary = safe_numeric_summary(synth_series)

        key = (dataset_id, feature)

        row = {
            "dataset_id": dataset_id,
            "feature": feature,
            "feature_type": "numeric",
            "is_target": feature == TARGET_COLUMNS[dataset_id],
            "real_n": real_summary["valid_n"],
            "synthetic_n": synth_summary["valid_n"],
            "real_total_n": int(len(real_series)),
            "synthetic_total_n": int(len(synth_series)),
            "real_missing_count": int(real_series.isna().sum()),
            "synthetic_missing_count": int(synth_series.isna().sum()),
            "real_missing_rate": float(real_series.isna().mean()),
            "synthetic_missing_rate": float(synth_series.isna().mean()),
            "real_mean": real_summary["mean"],
            "synthetic_mean": synth_summary["mean"],
            "real_std": real_summary["std"],
            "synthetic_std": synth_summary["std"],
            "real_min": real_summary["min"],
            "synthetic_min": synth_summary["min"],
            "real_q25": real_summary["q25"],
            "synthetic_q25": synth_summary["q25"],
            "real_median": real_summary["median"],
            "synthetic_median": synth_summary["median"],
            "real_q75": real_summary["q75"],
            "synthetic_q75": synth_summary["q75"],
            "real_max": real_summary["max"],
            "synthetic_max": synth_summary["max"],
        }

        # Preserve full quantile vectors for later distribution-distance metrics.
        for source_name, values in [
            ("real", real_values),
            ("synthetic", synth_values),
        ]:
            if len(values) >= MIN_VALID_NUMERIC_ROWS:
                quantiles = np.quantile(values, QUANTILE_LEVELS)
            else:
                quantiles = np.array([], dtype=float)

            NUMERIC_QUANTILES[(dataset_id, feature, source_name)] = quantiles

        # Store finite-value counts and missingness alongside histogram inputs.
        # Use a shared range so real and synthetic histograms are comparable.
        if len(real_values) > 0 and len(synth_values) > 0:
            combined_min = float(min(real_values.min(), synth_values.min()))
            combined_max = float(max(real_values.max(), synth_values.max()))

            if combined_min == combined_max:
                padding = max(abs(combined_min) * 0.01, 0.5)
                histogram_range = (
                    combined_min - padding,
                    combined_max + padding,
                )
            else:
                histogram_range = (combined_min, combined_max)

            real_counts, histogram_edges = np.histogram(
                real_values,
                bins=HISTOGRAM_BINS,
                range=histogram_range,
            )
            synth_counts, _ = np.histogram(
                synth_values,
                bins=HISTOGRAM_BINS,
                range=histogram_range,
            )

            NUMERIC_HISTOGRAMS[key] = {
                "bin_edges": histogram_edges,
                "real_counts": real_counts,
                "synthetic_counts": synth_counts,
                "real_valid_n": int(len(real_values)),
                "synthetic_valid_n": int(len(synth_values)),
                "range_min": histogram_range[0],
                "range_max": histogram_range[1],
            }
        else:
            NUMERIC_HISTOGRAMS[key] = {
                "bin_edges": np.array([], dtype=float),
                "real_counts": np.array([], dtype=int),
                "synthetic_counts": np.array([], dtype=int),
                "real_valid_n": int(len(real_values)),
                "synthetic_valid_n": int(len(synth_values)),
                "range_min": np.nan,
                "range_max": np.nan,
            }

        FEATURE_RESULT_MAP[key] = row

NUMERIC_SUMMARY_ROWS = list(FEATURE_RESULT_MAP.values())
NUMERIC_SUMMARY_DF = pd.DataFrame(NUMERIC_SUMMARY_ROWS)

NUMERIC_SUMMARY_PATH = NB14_FEATURE_ROOT / "numeric_distribution_summaries.csv"
NUMERIC_SUMMARY_DF.to_csv(NUMERIC_SUMMARY_PATH, index=False)

print("✓ Numerical summaries, quantile vectors, and shared-range histograms calculated.")
print("Numeric features evaluated:", sum(len(NUMERIC_FEATURES[d]) for d in DATASET_IDS))
print("Numeric summary rows:", len(NUMERIC_SUMMARY_DF))
print("Saved:", NUMERIC_SUMMARY_PATH)

display(NUMERIC_SUMMARY_DF.head(10))

✓ Numerical summaries, quantile vectors, and shared-range histograms calculated.
Numeric features evaluated: 24
Numeric summary rows: 24
Saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/feature_results/numeric_distribution_summaries.csv


,dataset_id,feature,feature_type,is_target,real_n,synthetic_n,real_total_n,synthetic_total_n,real_missing_count,synthetic_missing_count,...,real_min,synthetic_min,real_q25,synthetic_q25,real_median,synthetic_median,real_q75,synthetic_q75,real_max,synthetic_max
0,adult_income,age,numeric,False,7327,34189,7327,34189,0,0,...,17.0,-11.0,28.0,29.0,37.0,38.0,48.00,48.0,90.0,87.0
1,adult_income,fnlwgt,numeric,False,7327,34189,7327,34189,0,0,...,19214.0,-190453.0,119351.5,119991.0,180220.0,188730.0,239494.50,259889.0,1184622.0,601445.0
2,adult_income,education_num,numeric,False,7327,34189,7327,34189,0,0,...,1.0,1.0,9.0,8.0,10.0,10.0,12.00,12.0,16.0,20.0
3,adult_income,capital_gain,numeric,False,7327,34189,7327,34189,0,0,...,0.0,-12081.0,0.0,-1397.0,0.0,940.0,0.00,3397.0,99999.0,15131.0
4,adult_income,capital_loss,numeric,False,7327,34189,7327,34189,0,0,...,0.0,-1271.0,0.0,-161.0,0.0,93.0,0.00,362.0,4356.0,1849.0
5,adult_income,hours_per_week,numeric,False,7327,34189,7327,34189,0,0,...,1.0,-3.0,40.0,32.0,40.0,41.0,45.00,49.0,99.0,84.0
6,bank_marketing,age,numeric,False,6782,31647,6782,31647,0,0,...,18.0,5.0,33.0,33.0,39.0,40.0,49.00,48.0,93.0,82.0
7,bank_marketing,balance,numeric,False,6782,31647,6782,31647,0,0,...,-2604.0,-9285.0,70.0,-675.5,437.5,1050.0,1404.75,2884.0,81204.0,12186.0
8,bank_marketing,day,numeric,False,6782,31647,6782,31647,0,0,...,1.0,-14.0,8.0,11.0,16.0,16.0,21.00,22.0,31.0,46.0
9,bank_marketing,duration,numeric,False,6782,31647,6782,31647,0,0,...,0.0,-610.0,103.0,76.0,180.0,237.0,322.00,405.0,3183.0,1282.0


In [38]:
# =============================================================================
# SECTION 7 — CATEGORICAL DISTRIBUTION EVALUATION
# =============================================================================

def categorical_tokens(series):
    """Convert categorical values consistently while preserving missingness."""
    return series.astype("string").fillna("__MISSING__").astype(str)


def categorical_distribution(real_series, synth_series):
    """
    Build aligned categorical probability vectors from real and synthetic data.
    Apply a shared category cap only when necessary.
    """
    real_tokens = categorical_tokens(real_series)
    synth_tokens = categorical_tokens(synth_series)

    real_counts = real_tokens.value_counts(dropna=False)
    synth_counts = synth_tokens.value_counts(dropna=False)

    all_levels = sorted(set(real_counts.index) | set(synth_counts.index))

    category_cap_applied = len(all_levels) > MAX_CATEGORY_LEVELS

    if category_cap_applied:
        # Select retained categories using combined frequency in both datasets.
        combined_counts = real_counts.add(synth_counts, fill_value=0)
        keep_levels = set(
            combined_counts.nlargest(MAX_CATEGORY_LEVELS - 1).index
        )

        real_tokens = real_tokens.where(
            real_tokens.isin(keep_levels), "__OTHER__"
        )
        synth_tokens = synth_tokens.where(
            synth_tokens.isin(keep_levels), "__OTHER__"
        )

        real_counts = real_tokens.value_counts(dropna=False)
        synth_counts = synth_tokens.value_counts(dropna=False)

    levels = sorted(set(real_counts.index) | set(synth_counts.index))

    real_prob = real_counts.reindex(levels, fill_value=0).to_numpy(dtype=float)
    synth_prob = synth_counts.reindex(levels, fill_value=0).to_numpy(dtype=float)

    real_total = real_prob.sum()
    synth_total = synth_prob.sum()

    if real_total > 0:
        real_prob = real_prob / real_total

    if synth_total > 0:
        synth_prob = synth_prob / synth_total

    return levels, real_prob, synth_prob, category_cap_applied


CATEGORICAL_RESULT_ROWS = []

for dataset_id in DATASET_IDS:
    real_df = REAL_EVAL_DATA[dataset_id]
    synth_df = SYNTHETIC_EVAL_DATA[dataset_id]

    for feature in CATEGORICAL_FEATURES[dataset_id]:
        key = (dataset_id, feature)

        real_series = real_df[feature]
        synth_series = synth_df[feature]

        levels, real_prob, synth_prob, category_cap_applied = (
            categorical_distribution(real_series, synth_series)
        )

        CATEGORICAL_PROBABILITIES[(dataset_id, feature, "real")] = (
            levels, real_prob
        )
        CATEGORICAL_PROBABILITIES[(dataset_id, feature, "synthetic")] = (
            levels, synth_prob
        )

        real_tokens = categorical_tokens(real_series)
        synth_tokens = categorical_tokens(synth_series)

        # Report category overlap using original tokens, before any optional
        # high-cardinality grouping used for probability-vector computation.
        real_levels_original = set(real_tokens.unique())
        synth_levels_original = set(synth_tokens.unique())

        row = {
            "dataset_id": dataset_id,
            "feature": feature,
            "feature_type": "categorical",
            "is_target": feature == TARGET_COLUMNS[dataset_id],
            "real_n": int(len(real_series)),
            "synthetic_n": int(len(synth_series)),
            "real_valid_n": int(real_series.notna().sum()),
            "synthetic_valid_n": int(synth_series.notna().sum()),
            "real_missing_count": int(real_series.isna().sum()),
            "synthetic_missing_count": int(synth_series.isna().sum()),
            "real_missing_rate": float(real_series.isna().mean()),
            "synthetic_missing_rate": float(synth_series.isna().mean()),
            "real_category_count": int(real_levels_original.__len__()),
            "synthetic_category_count": int(synth_levels_original.__len__()),
            "shared_category_count": int(
                len(real_levels_original & synth_levels_original)
            ),
            "real_only_category_count": int(
                len(real_levels_original - synth_levels_original)
            ),
            "synthetic_only_category_count": int(
                len(synth_levels_original - real_levels_original)
            ),
            "aligned_category_count": int(len(levels)),
            "category_cap_applied": bool(category_cap_applied),
            "real_mean": np.nan,
            "synthetic_mean": np.nan,
            "real_std": np.nan,
            "synthetic_std": np.nan,
            "real_min": np.nan,
            "synthetic_min": np.nan,
            "real_median": np.nan,
            "synthetic_median": np.nan,
            "real_max": np.nan,
            "synthetic_max": np.nan,
        }

        FEATURE_RESULT_MAP[key] = row
        CATEGORICAL_RESULT_ROWS.append(row)

CATEGORICAL_SUMMARY_DF = pd.DataFrame(CATEGORICAL_RESULT_ROWS)

CATEGORICAL_SUMMARY_PATH = (
    NB14_FEATURE_ROOT / "categorical_distribution_summaries.csv"
)
CATEGORICAL_SUMMARY_DF.to_csv(CATEGORICAL_SUMMARY_PATH, index=False)

print("✓ Categorical probability vectors calculated and aligned.")
print(
    "Categorical features evaluated:",
    sum(len(CATEGORICAL_FEATURES[d]) for d in DATASET_IDS),
)
print("Categorical summary rows:", len(CATEGORICAL_SUMMARY_DF))
print("Category-cap applications:", int(
    CATEGORICAL_SUMMARY_DF["category_cap_applied"].sum()
))
print("Saved:", CATEGORICAL_SUMMARY_PATH)

display(CATEGORICAL_SUMMARY_DF.head(10))

✓ Categorical probability vectors calculated and aligned.
Categorical features evaluated: 56
Categorical summary rows: 56
Category-cap applications: 0
Saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/feature_results/categorical_distribution_summaries.csv


,dataset_id,feature,feature_type,is_target,real_n,synthetic_n,real_valid_n,synthetic_valid_n,real_missing_count,synthetic_missing_count,...,real_mean,synthetic_mean,real_std,synthetic_std,real_min,synthetic_min,real_median,synthetic_median,real_max,synthetic_max
0,adult_income,workclass,categorical,False,7327,34189,6948,34189,379,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,adult_income,education,categorical,False,7327,34189,7327,34189,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,adult_income,marital_status,categorical,False,7327,34189,7327,34189,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,adult_income,occupation,categorical,False,7327,34189,6945,34189,382,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,adult_income,relationship,categorical,False,7327,34189,7327,34189,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,adult_income,race,categorical,False,7327,34189,7327,34189,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,adult_income,sex,categorical,False,7327,34189,7327,34189,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,adult_income,native_country,categorical,False,7327,34189,7197,34189,130,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,adult_income,income,categorical,True,7327,34189,7327,34189,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,bank_marketing,job,categorical,False,6782,31647,6782,31647,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
# =============================================================================
# SECTION 8 — MAE (DISTRIBUTION-SUMMARY MAE)
# =============================================================================

def mae(a, b):
    """Calculate MAE for aligned, finite, equal-length numeric vectors."""
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    if len(a) == 0 or len(a) != len(b):
        return np.nan

    valid = np.isfinite(a) & np.isfinite(b)

    if not valid.any():
        return np.nan

    return float(np.mean(np.abs(a[valid] - b[valid])))


for (dataset_id, feature), row in FEATURE_RESULT_MAP.items():
    if row["feature_type"] == "numeric":
        real_vec = NUMERIC_QUANTILES.get(
            (dataset_id, feature, "real"), np.array([])
        )
        synth_vec = NUMERIC_QUANTILES.get(
            (dataset_id, feature, "synthetic"), np.array([])
        )

        if len(real_vec) != len(synth_vec) or len(real_vec) == 0:
            row["mae_distribution_summary"] = np.nan
        else:
            row["mae_distribution_summary"] = mae(real_vec, synth_vec)

        row["mae_definition"] = "Mean absolute difference between aligned quantiles"

    else:
        real_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "real")
        )
        synth_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "synthetic")
        )

        if real_entry is None or synth_entry is None:
            row["mae_distribution_summary"] = np.nan
        else:
            real_levels, real_vec = real_entry
            synth_levels, synth_vec = synth_entry

            if list(real_levels) != list(synth_levels):
                row["mae_distribution_summary"] = np.nan
            else:
                row["mae_distribution_summary"] = mae(
                    real_vec, synth_vec
                )

        row["mae_definition"] = "Mean absolute difference between aligned category probabilities"

print("✓ Distribution-summary MAE calculated.")
print(
    "Valid MAE results:",
    int(pd.to_numeric(
        pd.Series([
            row.get("mae_distribution_summary", np.nan)
            for row in FEATURE_RESULT_MAP.values()
        ]),
        errors="coerce",
    ).notna().sum()),
)
print("✓ Numeric quantiles and categorical probability vectors checked for alignment.")

✓ Distribution-summary MAE calculated.
Valid MAE results: 80
✓ Numeric quantiles and categorical probability vectors checked for alignment.


In [42]:
# =============================================================================
# SECTION 9 — RMSE (DISTRIBUTION-SUMMARY RMSE)
# =============================================================================

def rmse(a, b):
    """Calculate RMSE for aligned, finite, equal-length vectors."""
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    if len(a) == 0 or len(a) != len(b):
        return np.nan

    valid = np.isfinite(a) & np.isfinite(b)

    if not valid.any():
        return np.nan

    return float(np.sqrt(np.mean(np.square(a[valid] - b[valid]))))


for (dataset_id, feature), row in FEATURE_RESULT_MAP.items():
    if row["feature_type"] == "numeric":
        real_vec = NUMERIC_QUANTILES.get(
            (dataset_id, feature, "real"), np.array([])
        )
        synth_vec = NUMERIC_QUANTILES.get(
            (dataset_id, feature, "synthetic"), np.array([])
        )
        metric_definition = "Root mean squared difference between aligned quantiles"

    else:
        real_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "real")
        )
        synth_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "synthetic")
        )

        if real_entry is None or synth_entry is None:
            real_vec = np.array([])
            synth_vec = np.array([])
        else:
            real_levels, real_vec = real_entry
            synth_levels, synth_vec = synth_entry

            if list(real_levels) != list(synth_levels):
                real_vec = np.array([])
                synth_vec = np.array([])

        metric_definition = "Root mean squared difference between aligned category probabilities"

    row["rmse_distribution_summary"] = rmse(real_vec, synth_vec)
    row["rmse_definition"] = metric_definition

rmse_values = pd.to_numeric(
    pd.Series([
        row.get("rmse_distribution_summary", np.nan)
        for row in FEATURE_RESULT_MAP.values()
    ]),
    errors="coerce",
)

print("✓ Distribution-summary RMSE calculated.")
print("Total feature results:", len(FEATURE_RESULT_MAP))
print("Valid RMSE results:", int(rmse_values.notna().sum()))

✓ Distribution-summary RMSE calculated.
Total feature results: 80
Valid RMSE results: 80


In [44]:
# =============================================================================
# SECTION 10 — R² (DISTRIBUTION-SUMMARY R²)
# =============================================================================

def r2_summary(a, b):
    """
    Calculate descriptive R² between aligned distribution summaries.
    This is not predictive-model R².
    """
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    if len(a) < 2 or len(a) != len(b):
        return np.nan

    valid = np.isfinite(a) & np.isfinite(b)
    a = a[valid]
    b = b[valid]

    if len(a) < 2:
        return np.nan

    denominator = float(np.sum((a - np.mean(a)) ** 2))

    if denominator <= 1e-15:
        return np.nan

    return float(1.0 - np.sum((a - b) ** 2) / denominator)


for (dataset_id, feature), row in FEATURE_RESULT_MAP.items():
    if row["feature_type"] == "numeric":
        real_vec = NUMERIC_QUANTILES.get(
            (dataset_id, feature, "real"), np.array([])
        )
        synth_vec = NUMERIC_QUANTILES.get(
            (dataset_id, feature, "synthetic"), np.array([])
        )
    else:
        real_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "real")
        )
        synth_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "synthetic")
        )

        if real_entry is None or synth_entry is None:
            real_vec = np.array([])
            synth_vec = np.array([])
        else:
            real_levels, real_vec = real_entry
            synth_levels, synth_vec = synth_entry

            if list(real_levels) != list(synth_levels):
                real_vec = np.array([])
                synth_vec = np.array([])

    row["r2_distribution_summary"] = r2_summary(real_vec, synth_vec)
    row["r2_definition"] = (
        "Descriptive R² between aligned distribution summaries; "
        "not predictive-model R²"
    )

r2_values = pd.to_numeric(
    pd.Series([
        row.get("r2_distribution_summary", np.nan)
        for row in FEATURE_RESULT_MAP.values()
    ]),
    errors="coerce",
)

print("✓ Distribution-summary R² calculated.")
print("Total feature results:", len(FEATURE_RESULT_MAP))
print("Valid R² results:", int(r2_values.notna().sum()))
print("Undefined R² results:", int(r2_values.isna().sum()))

✓ Distribution-summary R² calculated.
Total feature results: 80
Valid R² results: 77
Undefined R² results: 3


In [45]:
# =============================================================================
# SECTION 11 — WASSERSTEIN DISTANCE
# =============================================================================

from scipy.stats import wasserstein_distance

for (dataset_id, feature), row in FEATURE_RESULT_MAP.items():
    row["wasserstein_distance"] = np.nan

    if row["feature_type"] != "numeric":
        continue

    real_values = numeric_array(REAL_EVAL_DATA[dataset_id][feature])
    synth_values = numeric_array(SYNTHETIC_EVAL_DATA[dataset_id][feature])

    if len(real_values) and len(synth_values):
        row["wasserstein_distance"] = float(
            wasserstein_distance(real_values, synth_values)
        )

print("✓ Wasserstein distance calculated for numeric features.")

✓ Wasserstein distance calculated for numeric features.


In [47]:
# =============================================================================
# SECTION 12 — JS DIVERGENCE
# =============================================================================

from scipy.spatial.distance import jensenshannon


def smooth_probability(prob, smoothing=PROBABILITY_SMOOTHING):
    """Apply additive smoothing and normalize a probability vector."""
    prob = np.asarray(prob, dtype=float).ravel()

    if len(prob) == 0 or not np.all(np.isfinite(prob)):
        return np.array([], dtype=float)

    if np.any(prob < 0):
        return np.array([], dtype=float)

    prob = prob + smoothing
    total = prob.sum()

    if not np.isfinite(total) or total <= 0:
        return np.array([], dtype=float)

    return prob / total


def js_divergence_from_probabilities(p, q):
    """Calculate base-2 Jensen–Shannon divergence, not its square root."""
    p = np.asarray(p, dtype=float).ravel()
    q = np.asarray(q, dtype=float).ravel()

    if len(p) == 0 or len(p) != len(q):
        return np.nan

    p = smooth_probability(p)
    q = smooth_probability(q)

    if len(p) == 0 or len(q) == 0:
        return np.nan

    # scipy's Jensen–Shannon function returns the distance (square root
    # of divergence); squaring produces JS divergence in [0, 1] for base 2.
    distance = jensenshannon(p, q, base=2)

    if not np.isfinite(distance):
        return np.nan

    return float(np.clip(distance ** 2, 0.0, 1.0))


def numeric_histogram_probabilities(real_values, synth_values):
    """
    Build comparable histogram probabilities using shared bin edges.
    Non-finite values are excluded from distribution bins and remain
    represented by the missingness metrics from earlier sections.
    """
    real_values = np.asarray(real_values, dtype=float)
    synth_values = np.asarray(synth_values, dtype=float)

    real_values = real_values[np.isfinite(real_values)]
    synth_values = synth_values[np.isfinite(synth_values)]

    if len(real_values) == 0 or len(synth_values) == 0:
        return np.array([], dtype=float), np.array([], dtype=float)

    combined = np.concatenate([real_values, synth_values])
    lo = float(np.min(combined))
    hi = float(np.max(combined))

    if np.isclose(lo, hi):
        return np.array([1.0]), np.array([1.0])

    edges = np.linspace(lo, hi, HISTOGRAM_BINS + 1)

    real_counts, _ = np.histogram(real_values, bins=edges)
    synth_counts, _ = np.histogram(synth_values, bins=edges)

    if real_counts.sum() == 0 or synth_counts.sum() == 0:
        return np.array([], dtype=float), np.array([], dtype=float)

    return (
        real_counts.astype(float) / real_counts.sum(),
        synth_counts.astype(float) / synth_counts.sum(),
    )


for (dataset_id, feature), row in FEATURE_RESULT_MAP.items():
    if row["feature_type"] == "numeric":
        real_values = numeric_array(REAL_EVAL_DATA[dataset_id][feature])
        synth_values = numeric_array(SYNTHETIC_EVAL_DATA[dataset_id][feature])

        p, q = numeric_histogram_probabilities(real_values, synth_values)

    else:
        real_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "real")
        )
        synth_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "synthetic")
        )

        if real_entry is None or synth_entry is None:
            p, q = np.array([]), np.array([])
        else:
            real_levels, p = real_entry
            synth_levels, q = synth_entry

            if list(real_levels) != list(synth_levels):
                p, q = np.array([]), np.array([])

    row["js_divergence"] = js_divergence_from_probabilities(p, q)

print("✓ Jensen–Shannon divergence calculated.")
print(
    "Valid JS results:",
    int(pd.to_numeric(
        pd.Series([
            row.get("js_divergence", np.nan)
            for row in FEATURE_RESULT_MAP.values()
        ]),
        errors="coerce",
    ).notna().sum()),
)

✓ Jensen–Shannon divergence calculated.
Valid JS results: 80


In [49]:
# =============================================================================
# SECTION 13 — KL DIVERGENCE
# =============================================================================

def kl_divergence_from_probabilities(p, q):
    """
    Calculate directional KL divergence D_KL(P || Q).
    Probability vectors must represent the same aligned categories/bins.
    """
    p = np.asarray(p, dtype=float).ravel()
    q = np.asarray(q, dtype=float).ravel()

    if len(p) == 0 or len(p) != len(q):
        return np.nan

    p = smooth_probability(p)
    q = smooth_probability(q)

    if len(p) == 0 or len(q) == 0:
        return np.nan

    if not np.all(np.isfinite(p)) or not np.all(np.isfinite(q)):
        return np.nan

    if np.any(p <= 0) or np.any(q <= 0):
        return np.nan

    return float(np.sum(p * np.log(p / q)))


for (dataset_id, feature), row in FEATURE_RESULT_MAP.items():
    if row["feature_type"] == "numeric":
        real_values = numeric_array(REAL_EVAL_DATA[dataset_id][feature])
        synth_values = numeric_array(SYNTHETIC_EVAL_DATA[dataset_id][feature])

        p, q = numeric_histogram_probabilities(real_values, synth_values)

    else:
        real_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "real")
        )
        synth_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "synthetic")
        )

        if real_entry is None or synth_entry is None:
            p, q = np.array([]), np.array([])
        else:
            real_levels, p = real_entry
            synth_levels, q = synth_entry

            if list(real_levels) != list(synth_levels):
                p, q = np.array([]), np.array([])

    row["kl_real_to_synthetic"] = kl_divergence_from_probabilities(p, q)
    row["kl_synthetic_to_real"] = kl_divergence_from_probabilities(q, p)
    row["kl_definition"] = (
        "Directional KL divergence with additive probability smoothing; "
        "natural logarithm"
    )

kl_real_to_synthetic_values = pd.to_numeric(
    pd.Series([
        row.get("kl_real_to_synthetic", np.nan)
        for row in FEATURE_RESULT_MAP.values()
    ]),
    errors="coerce",
)

kl_synthetic_to_real_values = pd.to_numeric(
    pd.Series([
        row.get("kl_synthetic_to_real", np.nan)
        for row in FEATURE_RESULT_MAP.values()
    ]),
    errors="coerce",
)

print("✓ Directional KL divergences calculated.")
print(
    "Valid real-to-synthetic KL results:",
    int(kl_real_to_synthetic_values.notna().sum()),
)
print(
    "Valid synthetic-to-real KL results:",
    int(kl_synthetic_to_real_values.notna().sum()),
)

✓ Directional KL divergences calculated.
Valid real-to-synthetic KL results: 80
Valid synthetic-to-real KL results: 80


In [51]:
# =============================================================================
# SECTION 14 — PEARSON CORRELATION
# =============================================================================

from scipy.stats import pearsonr, spearmanr


def safe_vector_correlation(func, a, b):
    """Calculate correlation between aligned, finite summary vectors."""
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    if len(a) < 2 or len(a) != len(b):
        return np.nan

    valid = np.isfinite(a) & np.isfinite(b)
    a = a[valid]
    b = b[valid]

    if len(a) < 2:
        return np.nan

    if np.std(a) <= 1e-15 or np.std(b) <= 1e-15:
        return np.nan

    try:
        result = func(a, b)
        statistic = result.statistic if hasattr(result, "statistic") else result[0]

        if not np.isfinite(statistic):
            return np.nan

        return float(np.clip(statistic, -1.0, 1.0))

    except (ValueError, TypeError, FloatingPointError):
        return np.nan


for (dataset_id, feature), row in FEATURE_RESULT_MAP.items():
    if row["feature_type"] == "numeric":
        real_vec = NUMERIC_QUANTILES.get(
            (dataset_id, feature, "real"), np.array([])
        )
        synth_vec = NUMERIC_QUANTILES.get(
            (dataset_id, feature, "synthetic"), np.array([])
        )

    else:
        real_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "real")
        )
        synth_entry = CATEGORICAL_PROBABILITIES.get(
            (dataset_id, feature, "synthetic")
        )

        if real_entry is None or synth_entry is None:
            real_vec = np.array([])
            synth_vec = np.array([])
        else:
            real_levels, real_vec = real_entry
            synth_levels, synth_vec = synth_entry

            if list(real_levels) != list(synth_levels):
                real_vec = np.array([])
                synth_vec = np.array([])

    row["pearson_distribution_correlation"] = safe_vector_correlation(
        pearsonr, real_vec, synth_vec
    )

    row["pearson_definition"] = (
        "Pearson correlation between aligned distribution-summary vectors; "
        "not feature-to-feature dependency preservation"
    )

pearson_values = pd.to_numeric(
    pd.Series([
        row.get("pearson_distribution_correlation", np.nan)
        for row in FEATURE_RESULT_MAP.values()
    ]),
    errors="coerce",
)

print("✓ Pearson distribution-summary correlations calculated.")
print("Total feature results:", len(FEATURE_RESULT_MAP))
print("Valid Pearson results:", int(pearson_values.notna().sum()))
print("Undefined Pearson results:", int(pearson_values.isna().sum()))

✓ Pearson distribution-summary correlations calculated.
Total feature results: 80
Valid Pearson results: 77
Undefined Pearson results: 3


In [52]:
# =============================================================================
# SECTION 15 — SPEARMAN CORRELATION
# =============================================================================

for (dataset_id, feature), row in FEATURE_RESULT_MAP.items():
    if row["feature_type"] == "numeric":
        real_vec = NUMERIC_QUANTILES[(dataset_id, feature, "real")]
        synth_vec = NUMERIC_QUANTILES[(dataset_id, feature, "synthetic")]
    else:
        _, real_vec = CATEGORICAL_PROBABILITIES[
            (dataset_id, feature, "real")
        ]
        _, synth_vec = CATEGORICAL_PROBABILITIES[
            (dataset_id, feature, "synthetic")
        ]

    row["spearman_distribution_correlation"] = safe_vector_correlation(
        spearmanr, real_vec, synth_vec
    )

print("✓ Spearman distribution-summary correlations calculated.")

✓ Spearman distribution-summary correlations calculated.


In [54]:
# =============================================================================
# SECTION 16 — DEPENDENCY PRESERVATION
# =============================================================================

from scipy.stats import chi2_contingency

DEPENDENCY_ROWS = []
DEPENDENCY_SUMMARY_ROWS = []


def numeric_correlation_matrix(df, columns, method):
    """Calculate pairwise numeric correlations after excluding invalid columns."""
    columns = [col for col in columns if col in df.columns]

    if len(columns) < 2:
        return pd.DataFrame()

    work = df[columns].apply(pd.to_numeric, errors="coerce")
    work = work.replace([np.inf, -np.inf], np.nan)

    valid_columns = [
        col for col in work.columns
        if work[col].notna().sum() >= 2
        and work[col].nunique(dropna=True) > 1
    ]

    if len(valid_columns) < 2:
        return pd.DataFrame()

    return work[valid_columns].corr(method=method, min_periods=2)


def cramers_v(x, y):
    """
    Calculate bias-corrected Cramér's V for categorical association.
    Missing values are represented explicitly as a category.
    """
    frame = pd.DataFrame({
        "x": categorical_tokens(x),
        "y": categorical_tokens(y),
    })

    table = pd.crosstab(frame["x"], frame["y"])

    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan

    n = int(table.to_numpy().sum())

    if n <= 1:
        return np.nan

    try:
        chi2 = chi2_contingency(table, correction=False)[0]
        phi2 = chi2 / n
        r, k = table.shape

        phi2_corrected = max(
            0.0,
            phi2 - ((k - 1) * (r - 1)) / (n - 1),
        )

        r_corrected = r - ((r - 1) ** 2) / (n - 1)
        k_corrected = k - ((k - 1) ** 2) / (n - 1)

        denominator = min(k_corrected - 1, r_corrected - 1)

        if denominator <= 0:
            return np.nan

        value = np.sqrt(phi2_corrected / denominator)

        return float(np.clip(value, 0.0, 1.0))

    except (ValueError, ZeroDivisionError, FloatingPointError):
        return np.nan


def append_dependency_summary(
    dataset_id,
    dependency_type,
    features_compared,
    real_values,
    synthetic_values,
):
    """Summarize aligned real-versus-synthetic association values."""
    real_values = np.asarray(real_values, dtype=float).ravel()
    synthetic_values = np.asarray(synthetic_values, dtype=float).ravel()

    if len(real_values) != len(synthetic_values):
        raise ValueError(
            f"{dataset_id} {dependency_type}: association vectors are misaligned."
        )

    valid = np.isfinite(real_values) & np.isfinite(synthetic_values)
    real_valid = real_values[valid]
    synthetic_valid = synthetic_values[valid]

    if len(real_valid):
        differences = real_valid - synthetic_valid
        dependency_mae = float(np.mean(np.abs(differences)))
        dependency_rmse = float(np.sqrt(np.mean(differences ** 2)))
        dependency_status = "PASS"
    else:
        dependency_mae = np.nan
        dependency_rmse = np.nan
        dependency_status = "REVIEW"

    DEPENDENCY_SUMMARY_ROWS.append({
        "dataset_id": dataset_id,
        "dependency_type": dependency_type,
        "features_compared": int(features_compared),
        "pairs_compared": int(valid.sum()),
        "pairs_available": int(len(real_values)),
        "dependency_mae": dependency_mae,
        "dependency_rmse": dependency_rmse,
        "dependency_status": dependency_status,
    })


for dataset_id in DATASET_IDS:
    real_df = REAL_EVAL_DATA[dataset_id]
    synth_df = SYNTHETIC_EVAL_DATA[dataset_id]
    target = TARGET_COLUMNS[dataset_id]

    # -------------------------------------------------------------------------
    # 1. Numeric feature dependency preservation
    # -------------------------------------------------------------------------
    numeric_cols = [
        col for col in NUMERIC_FEATURES[dataset_id]
        if col in real_df.columns and col in synth_df.columns
    ]

    for method in ["pearson", "spearman"]:
        real_corr = numeric_correlation_matrix(
            real_df, numeric_cols, method
        )
        synth_corr = numeric_correlation_matrix(
            synth_df, numeric_cols, method
        )

        # Use only features valid in both correlation matrices.
        shared = [
            col for col in real_corr.columns
            if col in synth_corr.columns
        ]

        if len(shared) < 2:
            append_dependency_summary(
                dataset_id=dataset_id,
                dependency_type=f"numeric_{method}_correlation",
                features_compared=len(shared),
                real_values=[],
                synthetic_values=[],
            )
            continue

        real_mat = real_corr.loc[shared, shared].to_numpy(dtype=float)
        synth_mat = synth_corr.loc[shared, shared].to_numpy(dtype=float)

        # Evaluate unique off-diagonal feature pairs only.
        upper_i, upper_j = np.triu_indices(len(shared), k=1)
        real_values = real_mat[upper_i, upper_j]
        synth_values = synth_mat[upper_i, upper_j]

        valid = np.isfinite(real_values) & np.isfinite(synth_values)

        append_dependency_summary(
            dataset_id=dataset_id,
            dependency_type=f"numeric_{method}_correlation",
            features_compared=len(shared),
            real_values=real_values,
            synthetic_values=synth_values,
        )

        for i, j in zip(upper_i, upper_j):
            real_assoc = real_mat[i, j]
            synth_assoc = synth_mat[i, j]

            if not (
                np.isfinite(real_assoc)
                and np.isfinite(synth_assoc)
            ):
                continue

            DEPENDENCY_ROWS.append({
                "dataset_id": dataset_id,
                "dependency_type": f"numeric_{method}_correlation",
                "feature_1": shared[i],
                "feature_2": shared[j],
                "real_association": float(real_assoc),
                "synthetic_association": float(synth_assoc),
                "absolute_association_difference": float(
                    abs(real_assoc - synth_assoc)
                ),
                "association_status": "VALID",
            })

    # -------------------------------------------------------------------------
    # 2. Categorical feature-to-target dependency preservation
    # -------------------------------------------------------------------------
    categorical_features = [
        col for col in CATEGORICAL_FEATURES[dataset_id]
        if col != target
        and col in real_df.columns
        and col in synth_df.columns
    ]

    target_real_associations = []
    target_synthetic_associations = []

    for feature in categorical_features:
        real_assoc = cramers_v(real_df[feature], real_df[target])
        synth_assoc = cramers_v(synth_df[feature], synth_df[target])

        valid_associations = (
            np.isfinite(real_assoc) and np.isfinite(synth_assoc)
        )

        DEPENDENCY_ROWS.append({
            "dataset_id": dataset_id,
            "dependency_type": "categorical_target_cramers_v",
            "feature_1": feature,
            "feature_2": target,
            "real_association": (
                float(real_assoc) if np.isfinite(real_assoc) else np.nan
            ),
            "synthetic_association": (
                float(synth_assoc) if np.isfinite(synth_assoc) else np.nan
            ),
            "absolute_association_difference": (
                float(abs(real_assoc - synth_assoc))
                if valid_associations
                else np.nan
            ),
            "association_status": (
                "VALID" if valid_associations else "UNDEFINED"
            ),
        })

        target_real_associations.append(real_assoc)
        target_synthetic_associations.append(synth_assoc)

    append_dependency_summary(
        dataset_id=dataset_id,
        dependency_type="categorical_target_cramers_v",
        features_compared=len(categorical_features),
        real_values=target_real_associations,
        synthetic_values=target_synthetic_associations,
    )


# =============================================================================
# BUILD AND SAVE DEPENDENCY RESULTS
# =============================================================================

DEPENDENCY_DF = pd.DataFrame(DEPENDENCY_ROWS)
DEPENDENCY_SUMMARY_DF = pd.DataFrame(DEPENDENCY_SUMMARY_ROWS)

DEPENDENCY_DETAILS_PATH = (
    NB14_FEATURE_ROOT / "dependency_preservation_details.csv"
)
DEPENDENCY_SUMMARY_PATH = (
    NB14_DATASET_ROOT / "dependency_preservation_summary.csv"
)

DEPENDENCY_DF.to_csv(DEPENDENCY_DETAILS_PATH, index=False)
DEPENDENCY_SUMMARY_DF.to_csv(DEPENDENCY_SUMMARY_PATH, index=False)

display(DEPENDENCY_SUMMARY_DF)

print("Detailed dependency rows:", len(DEPENDENCY_DF))
print(
    "Valid detailed associations:",
    int((DEPENDENCY_DF["association_status"] == "VALID").sum())
    if not DEPENDENCY_DF.empty else 0,
)
print("Dependency details saved:", DEPENDENCY_DETAILS_PATH)
print("Dependency summary saved:", DEPENDENCY_SUMMARY_PATH)

,dataset_id,dependency_type,features_compared,pairs_compared,pairs_available,dependency_mae,dependency_rmse,dependency_status
0,adult_income,numeric_pearson_correlation,6,15,15,0.045097,0.052013,PASS
1,adult_income,numeric_spearman_correlation,6,15,15,0.059140,0.070269,PASS
2,adult_income,categorical_target_cramers_v,8,8,8,0.093528,0.136120,PASS
3,bank_marketing,numeric_pearson_correlation,7,21,21,0.033397,0.045383,PASS
4,bank_marketing,numeric_spearman_correlation,7,21,21,0.070741,0.133674,PASS
5,bank_marketing,categorical_target_cramers_v,9,9,9,0.037784,0.049081,PASS
6,diabetes_130us,numeric_pearson_correlation,11,55,55,0.120329,0.159246,PASS
7,diabetes_130us,numeric_spearman_correlation,11,55,55,0.144089,0.185596,PASS
8,diabetes_130us,categorical_target_cramers_v,36,25,36,0.052181,0.069311,PASS


Detailed dependency rows: 235
Valid detailed associations: 224
Dependency details saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/feature_results/dependency_preservation_details.csv
Dependency summary saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/dataset_results/dependency_preservation_summary.csv


In [57]:
# =============================================================================
# SECTION 17 — FEATURE-LEVEL RESULTS
# =============================================================================

FEATURE_RESULTS_DF = pd.DataFrame(
    list(FEATURE_RESULT_MAP.values())
)

if FEATURE_RESULTS_DF.empty:
    raise ValueError(
        "Feature-level results are empty. Run Sections 6–15 first."
    )

# -----------------------------------------------------------------------------
# 1. Validate feature-level coverage and uniqueness
# -----------------------------------------------------------------------------

required_columns = [
    "dataset_id",
    "feature",
    "feature_type",
    "is_target",
]

missing_required = [
    col for col in required_columns
    if col not in FEATURE_RESULTS_DF.columns
]

if missing_required:
    raise ValueError(
        f"Required columns missing: {missing_required}"
    )

if FEATURE_RESULTS_DF.duplicated(["dataset_id", "feature"]).any():
    raise ValueError(
        "Duplicate dataset-feature combinations detected."
    )

expected_feature_counts = {
    dataset_id: (
        len(NUMERIC_FEATURES[dataset_id])
        + len(CATEGORICAL_FEATURES[dataset_id])
    )
    for dataset_id in DATASET_IDS
}

for dataset_id in DATASET_IDS:
    observed = int(
        FEATURE_RESULTS_DF.loc[
            FEATURE_RESULTS_DF["dataset_id"] == dataset_id,
            "feature",
        ].nunique()
    )

    expected = expected_feature_counts[dataset_id]

    if observed != expected:
        raise ValueError(
            f"{dataset_id}: expected {expected} features, "
            f"observed {observed}."
        )

# -----------------------------------------------------------------------------
# 2. Arrange columns without discarding calculated metrics
# -----------------------------------------------------------------------------

preferred_columns = [
    "dataset_id", "feature", "feature_type", "is_target",
    "real_n", "synthetic_n", "real_total_n", "synthetic_total_n",
    "real_valid_n", "synthetic_valid_n",
    "real_missing_count", "synthetic_missing_count",
    "real_missing_rate", "synthetic_missing_rate",
    "real_category_count", "synthetic_category_count",
    "shared_category_count", "real_only_category_count",
    "synthetic_only_category_count", "aligned_category_count",
    "category_cap_applied",
    "real_mean", "synthetic_mean", "real_std", "synthetic_std",
    "real_min", "synthetic_min", "real_q25", "synthetic_q25",
    "real_median", "synthetic_median", "real_q75", "synthetic_q75",
    "real_max", "synthetic_max",
    "mae_distribution_summary", "mae_definition",
    "rmse_distribution_summary", "rmse_definition",
    "r2_distribution_summary", "r2_definition",
    "wasserstein_distance", "js_divergence",
    "kl_real_to_synthetic", "kl_synthetic_to_real", "kl_definition",
    "pearson_distribution_correlation", "pearson_definition",
    "spearman_distribution_correlation",
]

ordered_columns = [
    col for col in preferred_columns
    if col in FEATURE_RESULTS_DF.columns
]
additional_columns = [
    col for col in FEATURE_RESULTS_DF.columns
    if col not in ordered_columns
]

FEATURE_RESULTS_DF = (
    FEATURE_RESULTS_DF[ordered_columns + additional_columns]
    .sort_values(["dataset_id", "feature"], kind="stable")
    .reset_index(drop=True)
)

# -----------------------------------------------------------------------------
# 3. Validate metric ranges
# -----------------------------------------------------------------------------

bounded_metrics = {
    "js_divergence": (0.0, 1.0),
    "pearson_distribution_correlation": (-1.0, 1.0),
    "spearman_distribution_correlation": (-1.0, 1.0),
}

for metric, (lower, upper) in bounded_metrics.items():
    if metric not in FEATURE_RESULTS_DF.columns:
        continue

    values = pd.to_numeric(FEATURE_RESULTS_DF[metric], errors="coerce")
    values = values[np.isfinite(values)]

    if ((values < lower - 1e-8) | (values > upper + 1e-8)).any():
        raise ValueError(f"{metric} contains values outside [{lower}, {upper}].")

for metric in ["kl_real_to_synthetic", "kl_synthetic_to_real"]:
    if metric not in FEATURE_RESULTS_DF.columns:
        continue

    values = pd.to_numeric(FEATURE_RESULTS_DF[metric], errors="coerce")
    values = values[np.isfinite(values)]

    if (values < -1e-8).any():
        raise ValueError(f"{metric} contains negative KL values.")

# -----------------------------------------------------------------------------
# 4. Save feature-level fidelity results
# -----------------------------------------------------------------------------

FEATURE_RESULTS_PATH = (
    NB14_FEATURE_ROOT / "feature_level_fidelity_results.csv"
)

FEATURE_RESULTS_DF.to_csv(
    FEATURE_RESULTS_PATH,
    index=False,
    na_rep="",
)

# -----------------------------------------------------------------------------
# 5. Metric coverage using only applicable feature types
# -----------------------------------------------------------------------------

# Wasserstein distance is evaluated only for numeric features.
# All other metrics below are reported for the feature types evaluated
# by their corresponding calculations in Sections 8–15.
METRIC_APPLICABILITY = {
    "mae_distribution_summary": ["numeric", "categorical"],
    "rmse_distribution_summary": ["numeric", "categorical"],
    "r2_distribution_summary": ["numeric", "categorical"],
    "wasserstein_distance": ["numeric"],
    "js_divergence": ["numeric", "categorical"],
    "kl_real_to_synthetic": ["numeric", "categorical"],
    "kl_synthetic_to_real": ["numeric", "categorical"],
    "pearson_distribution_correlation": ["numeric", "categorical"],
    "spearman_distribution_correlation": ["numeric", "categorical"],
}

coverage_rows = []

for dataset_id in DATASET_IDS:
    dataset_results = FEATURE_RESULTS_DF[
        FEATURE_RESULTS_DF["dataset_id"] == dataset_id
    ]

    for metric, applicable_types in METRIC_APPLICABILITY.items():
        if metric not in dataset_results.columns:
            continue

        applicable = dataset_results[
            dataset_results["feature_type"].isin(applicable_types)
        ]

        values = pd.to_numeric(
            applicable[metric],
            errors="coerce",
        )

        coverage_rows.append({
            "dataset_id": dataset_id,
            "metric": metric,
            "applicable_feature_types": ", ".join(applicable_types),
            "applicable_feature_count": int(len(applicable)),
            "valid_count": int(values.notna().sum()),
            "undefined_count": int(values.isna().sum()),
            "coverage_rate": (
                float(values.notna().mean())
                if len(values) else np.nan
            ),
        })

FEATURE_METRIC_COVERAGE_DF = pd.DataFrame(coverage_rows)

FEATURE_METRIC_COVERAGE_PATH = (
    NB14_VALIDATION_ROOT / "feature_metric_coverage.csv"
)

FEATURE_METRIC_COVERAGE_DF.to_csv(
    FEATURE_METRIC_COVERAGE_PATH,
    index=False,
    na_rep="",
)

# -----------------------------------------------------------------------------
# 6. Completion summary
# -----------------------------------------------------------------------------

print("Feature-level rows:", len(FEATURE_RESULTS_DF))
print("Datasets evaluated:", FEATURE_RESULTS_DF["dataset_id"].nunique())
print("Unique dataset-feature pairs:",
      FEATURE_RESULTS_DF[["dataset_id", "feature"]].drop_duplicates().shape[0])
print("Expected feature counts:", expected_feature_counts)
print("Observed feature counts:",
      FEATURE_RESULTS_DF.groupby("dataset_id")["feature"].nunique().to_dict())
print("Saved feature results:", FEATURE_RESULTS_PATH)
print("Saved metric coverage:", FEATURE_METRIC_COVERAGE_PATH)

display(FEATURE_RESULTS_DF.head(20))
display(FEATURE_METRIC_COVERAGE_DF)

Feature-level rows: 80
Datasets evaluated: 3
Unique dataset-feature pairs: 80
Expected feature counts: {'adult_income': 15, 'bank_marketing': 17, 'diabetes_130us': 48}
Observed feature counts: {'adult_income': 15, 'bank_marketing': 17, 'diabetes_130us': 48}
Saved feature results: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/feature_results/feature_level_fidelity_results.csv
Saved metric coverage: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/feature_metric_coverage.csv


,dataset_id,feature,feature_type,is_target,real_n,synthetic_n,real_total_n,synthetic_total_n,real_valid_n,synthetic_valid_n,...,r2_distribution_summary,r2_definition,wasserstein_distance,js_divergence,kl_real_to_synthetic,kl_synthetic_to_real,kl_definition,pearson_distribution_correlation,pearson_definition,spearman_distribution_correlation
0,adult_income,age,numeric,False,7327,34189,7327.0,34189.0,NaN,NaN,...,0.945102,Descriptive R² between aligned distribution su...,1.423003,3.140580e-02,0.082223,0.507713,Directional KL divergence with additive probab...,0.973481,Pearson correlation between aligned distributi...,0.999534
1,adult_income,capital_gain,numeric,False,7327,34189,7327.0,34189.0,NaN,NaN,...,0.203105,Descriptive R² between aligned distribution su...,2977.181250,4.346911e-01,1.261376,5.535372,Directional KL divergence with additive probab...,0.460925,Pearson correlation between aligned distributi...,0.494195
2,adult_income,capital_loss,numeric,False,7327,34189,7327.0,34189.0,NaN,NaN,...,0.273792,Descriptive R² between aligned distribution su...,317.563639,7.295143e-01,2.463873,10.912757,Directional KL divergence with additive probab...,0.563122,Pearson correlation between aligned distributi...,0.375890
3,adult_income,education,categorical,False,7327,34189,NaN,NaN,7327.0,34189.0,...,0.994783,Descriptive R² between aligned distribution su...,NaN,7.910213e-04,0.002191,0.002198,Directional KL divergence with additive probab...,0.997912,Pearson correlation between aligned distributi...,0.988235
4,adult_income,education_num,numeric,False,7327,34189,7327.0,34189.0,NaN,NaN,...,0.897965,Descriptive R² between aligned distribution su...,0.561564,1.296441e-01,0.345391,0.498004,Directional KL divergence with additive probab...,0.955905,Pearson correlation between aligned distributi...,0.968444
5,adult_income,fnlwgt,numeric,False,7327,34189,7327.0,34189.0,NaN,NaN,...,0.785572,Descriptive R² between aligned distribution su...,13842.462725,3.196238e-02,0.116806,0.406361,Directional KL divergence with additive probab...,0.891480,Pearson correlation between aligned distributi...,1.000000
6,adult_income,hours_per_week,numeric,False,7327,34189,7327.0,34189.0,NaN,NaN,...,0.884499,Descriptive R² between aligned distribution su...,3.817932,3.040543e-01,0.986319,1.077589,Directional KL divergence with additive probab...,0.942223,Pearson correlation between aligned distributi...,0.947876
7,adult_income,income,categorical,True,7327,34189,NaN,NaN,7327.0,34189.0,...,0.987220,Descriptive R² between aligned distribution su...,NaN,9.004977e-04,0.002537,0.002459,Directional KL divergence with additive probab...,1.000000,Pearson correlation between aligned distributi...,1.000000
8,adult_income,marital_status,categorical,False,7327,34189,NaN,NaN,7327.0,34189.0,...,0.999599,Descriptive R² between aligned distribution su...,NaN,6.391546e-04,0.005186,0.001577,Directional KL divergence with additive probab...,0.999827,Pearson correlation between aligned distributi...,0.964286
9,adult_income,native_country,categorical,False,7327,34189,NaN,NaN,7197.0,34189.0,...,0.991524,Descriptive R² between aligned distribution su...,NaN,3.915920e-02,0.755598,0.080153,Directional KL divergence with additive probab...,0.999786,Pearson correlation between aligned distributi...,0.488400


,dataset_id,metric,applicable_feature_types,applicable_feature_count,valid_count,undefined_count,coverage_rate
0,adult_income,mae_distribution_summary,"numeric, categorical",15,15,0,1.0000
1,adult_income,rmse_distribution_summary,"numeric, categorical",15,15,0,1.0000
2,adult_income,r2_distribution_summary,"numeric, categorical",15,15,0,1.0000
3,adult_income,wasserstein_distance,numeric,6,6,0,1.0000
4,adult_income,js_divergence,"numeric, categorical",15,15,0,1.0000
5,adult_income,kl_real_to_synthetic,"numeric, categorical",15,15,0,1.0000
6,adult_income,kl_synthetic_to_real,"numeric, categorical",15,15,0,1.0000
7,adult_income,pearson_distribution_correlation,"numeric, categorical",15,15,0,1.0000
8,adult_income,spearman_distribution_correlation,"numeric, categorical",15,15,0,1.0000
9,bank_marketing,mae_distribution_summary,"numeric, categorical",17,17,0,1.0000


In [59]:
# =============================================================================
# SECTION 18 — DATASET-LEVEL AGGREGATION
# =============================================================================

print("=" * 100)
print("SECTION 18 — DATASET-LEVEL AGGREGATION")
print("=" * 100)

import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. Validate Required Inputs
# -----------------------------------------------------------------------------

required_columns = {
    "dataset_id",
    "feature",
    "feature_type",
    "real_n",
    "synthetic_n",
    *METRIC_COLUMNS,
}

missing_columns = required_columns.difference(FEATURE_RESULTS_DF.columns)

if missing_columns:
    raise ValueError(
        f"FEATURE_RESULTS_DF is missing required columns: "
        f"{sorted(missing_columns)}"
    )

if FEATURE_RESULTS_DF.empty:
    raise ValueError("FEATURE_RESULTS_DF is empty. Aggregation cannot proceed.")

if FEATURE_RESULTS_DF.duplicated(["dataset_id", "feature"]).any():
    duplicates = FEATURE_RESULTS_DF.loc[
        FEATURE_RESULTS_DF.duplicated(
            ["dataset_id", "feature"], keep=False
        ),
        ["dataset_id", "feature"],
    ]
    raise ValueError(
        "Duplicate dataset-feature rows detected:\n"
        f"{duplicates.to_string(index=False)}"
    )

# -----------------------------------------------------------------------------
# 2. Define Metric Applicability
# -----------------------------------------------------------------------------

# Wasserstein distance is evaluated only for numeric features.
# Other listed metrics use the aligned distribution summaries/vectors
# generated in the preceding sections.
METRIC_APPLICABILITY = {
    "mae_distribution_summary": {"numeric", "categorical"},
    "rmse_distribution_summary": {"numeric", "categorical"},
    "r2_distribution_summary": {"numeric", "categorical"},
    "wasserstein_distance": {"numeric"},
    "js_divergence": {"numeric", "categorical"},
    "kl_real_to_synthetic": {"numeric", "categorical"},
    "kl_synthetic_to_real": {"numeric", "categorical"},
    "pearson_distribution_correlation": {"numeric", "categorical"},
    "spearman_distribution_correlation": {"numeric", "categorical"},
}

# -----------------------------------------------------------------------------
# 3. Aggregate Metrics by Dataset and Feature Type
# -----------------------------------------------------------------------------

DATASET_AGGREGATION_ROWS = []
EXPECTED_FEATURE_COUNTS = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 48,
}

for dataset_id in DATASET_IDS:
    dataset_subset = FEATURE_RESULTS_DF.loc[
        FEATURE_RESULTS_DF["dataset_id"].eq(dataset_id)
    ].copy()

    if dataset_subset.empty:
        raise ValueError(
            f"No feature-level results found for dataset '{dataset_id}'."
        )

    expected_count = EXPECTED_FEATURE_COUNTS.get(dataset_id)

    if expected_count is not None and len(dataset_subset) != expected_count:
        raise ValueError(
            f"Unexpected feature count for {dataset_id}: "
            f"expected {expected_count}, observed {len(dataset_subset)}."
        )

    real_rows = len(REAL_EVAL_DATA[dataset_id])
    synthetic_rows = len(SYNTHETIC_EVAL_DATA[dataset_id])

    for feature_type in ("numeric", "categorical"):
        typed = dataset_subset.loc[
            dataset_subset["feature_type"].eq(feature_type)
        ].copy()

        row = {
            "dataset_id": dataset_id,
            "feature_type": feature_type,
            "feature_count": int(len(typed)),
            "real_rows": int(real_rows),
            "synthetic_rows": int(synthetic_rows),
            "target_column": TARGET_COLUMNS[dataset_id],
        }

        for metric in METRIC_COLUMNS:
            if metric not in typed.columns:
                continue

            applicable_types = METRIC_APPLICABILITY.get(
                metric, {"numeric", "categorical"}
            )

            is_applicable = feature_type in applicable_types

            if not is_applicable:
                row[f"mean_{metric}"] = np.nan
                row[f"median_{metric}"] = np.nan
                row[f"valid_{metric}_features"] = 0
                row[f"applicable_{metric}_features"] = 0
                row[f"undefined_{metric}_features"] = 0
                row[f"coverage_{metric}"] = np.nan
                continue

            values = pd.to_numeric(
                typed[metric], errors="coerce"
            ).replace([np.inf, -np.inf], np.nan)

            valid_values = values.dropna()
            applicable_count = int(len(typed))
            valid_count = int(len(valid_values))
            undefined_count = applicable_count - valid_count

            row[f"mean_{metric}"] = (
                float(valid_values.mean())
                if valid_count
                else np.nan
            )
            row[f"median_{metric}"] = (
                float(valid_values.median())
                if valid_count
                else np.nan
            )
            row[f"valid_{metric}_features"] = valid_count
            row[f"applicable_{metric}_features"] = applicable_count
            row[f"undefined_{metric}_features"] = undefined_count
            row[f"coverage_{metric}"] = (
                valid_count / applicable_count
                if applicable_count
                else np.nan
            )

        DATASET_AGGREGATION_ROWS.append(row)

# -----------------------------------------------------------------------------
# 4. Build and Validate Dataset-Level Results
# -----------------------------------------------------------------------------

DATASET_AGGREGATION_DF = pd.DataFrame(DATASET_AGGREGATION_ROWS)

if DATASET_AGGREGATION_DF.empty:
    raise ValueError("Dataset-level aggregation produced no rows.")

if DATASET_AGGREGATION_DF.duplicated(
    ["dataset_id", "feature_type"]
).any():
    raise ValueError(
        "Duplicate dataset-feature-type rows found in the aggregation."
    )

# Check that each dataset has both feature-type groups.
for dataset_id in DATASET_IDS:
    observed_types = set(
        DATASET_AGGREGATION_DF.loc[
            DATASET_AGGREGATION_DF["dataset_id"].eq(dataset_id),
            "feature_type",
        ]
    )

    if observed_types != {"numeric", "categorical"}:
        raise ValueError(
            f"Unexpected feature types for {dataset_id}: "
            f"{sorted(observed_types)}"
        )

# -----------------------------------------------------------------------------
# 5. Save Aggregation Artifact
# -----------------------------------------------------------------------------

NB14_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

aggregation_path = (
    NB14_DATASET_ROOT / "dataset_level_fidelity_aggregation.csv"
)

DATASET_AGGREGATION_DF.to_csv(
    aggregation_path,
    index=False,
    float_format="%.10g",
)

# -----------------------------------------------------------------------------
# 6. Display Results and Coverage Diagnostics
# -----------------------------------------------------------------------------

print(f"Aggregation rows: {len(DATASET_AGGREGATION_DF)}")
print(f"Datasets evaluated: {DATASET_AGGREGATION_DF['dataset_id'].nunique()}")
print(
    "Feature-type groups: "
    f"{DATASET_AGGREGATION_DF[['dataset_id', 'feature_type']].drop_duplicates().shape[0]}"
)
print(f"Saved aggregation: {aggregation_path}")

display(DATASET_AGGREGATION_DF)

coverage_columns = [
    "dataset_id",
    "feature_type",
    "feature_count",
]

for metric in METRIC_COLUMNS:
    coverage_columns.extend(
        [
            f"applicable_{metric}_features",
            f"valid_{metric}_features",
            f"undefined_{metric}_features",
            f"coverage_{metric}",
        ]
    )

coverage_columns = [
    column
    for column in coverage_columns
    if column in DATASET_AGGREGATION_DF.columns
]

DATASET_AGGREGATION_COVERAGE_DF = DATASET_AGGREGATION_DF[
    coverage_columns
].copy()

coverage_path = (
    NB14_VALIDATION_ROOT / "dataset_level_metric_coverage.csv"
)

NB14_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_AGGREGATION_COVERAGE_DF.to_csv(
    coverage_path,
    index=False,
    float_format="%.10g",
)

print(f"Saved metric coverage: {coverage_path}")
display(DATASET_AGGREGATION_COVERAGE_DF)

SECTION 18 — DATASET-LEVEL AGGREGATION
Aggregation rows: 6
Datasets evaluated: 3
Feature-type groups: 6
Saved aggregation: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/dataset_results/dataset_level_fidelity_aggregation.csv


,dataset_id,feature_type,feature_count,real_rows,synthetic_rows,target_column,mean_mae_distribution_summary,median_mae_distribution_summary,valid_mae_distribution_summary_features,applicable_mae_distribution_summary_features,...,valid_pearson_distribution_correlation_features,applicable_pearson_distribution_correlation_features,undefined_pearson_distribution_correlation_features,coverage_pearson_distribution_correlation,mean_spearman_distribution_correlation,median_spearman_distribution_correlation,valid_spearman_distribution_correlation_features,applicable_spearman_distribution_correlation_features,undefined_spearman_distribution_correlation_features,coverage_spearman_distribution_correlation
0,adult_income,numeric,6,7327,34189,income,3913.150495,172.560099,6,6,...,6,6,0,1.000000,0.797656,0.958160,6,6,0,1.000000
1,adult_income,categorical,9,7327,34189,income,0.008035,0.003985,9,9,...,9,9,0,1.000000,0.908526,0.988235,9,9,0,1.000000
2,bank_marketing,numeric,7,6782,31647,y,299.726011,3.702970,7,7,...,7,7,0,1.000000,0.899475,0.998630,7,7,0,1.000000
3,bank_marketing,categorical,10,6782,31647,y,0.025484,0.009211,10,10,...,10,10,0,1.000000,0.980000,1.000000,10,10,0,1.000000
4,diabetes_130us,numeric,11,15265,71236,readmitted,1.317957,0.861386,11,11,...,11,11,0,1.000000,0.859316,0.892363,11,11,0,1.000000
5,diabetes_130us,categorical,37,15265,71236,readmitted,0.037963,0.002009,37,37,...,34,37,3,0.918919,0.745646,1.000000,34,37,3,0.918919


Saved metric coverage: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/dataset_level_metric_coverage.csv


,dataset_id,feature_type,feature_count,applicable_mae_distribution_summary_features,valid_mae_distribution_summary_features,undefined_mae_distribution_summary_features,coverage_mae_distribution_summary,applicable_rmse_distribution_summary_features,valid_rmse_distribution_summary_features,undefined_rmse_distribution_summary_features,...,undefined_kl_synthetic_to_real_features,coverage_kl_synthetic_to_real,applicable_pearson_distribution_correlation_features,valid_pearson_distribution_correlation_features,undefined_pearson_distribution_correlation_features,coverage_pearson_distribution_correlation,applicable_spearman_distribution_correlation_features,valid_spearman_distribution_correlation_features,undefined_spearman_distribution_correlation_features,coverage_spearman_distribution_correlation
0,adult_income,numeric,6,6,6,0,1.0,6,6,0,...,0,1.0,6,6,0,1.000000,6,6,0,1.000000
1,adult_income,categorical,9,9,9,0,1.0,9,9,0,...,0,1.0,9,9,0,1.000000,9,9,0,1.000000
2,bank_marketing,numeric,7,7,7,0,1.0,7,7,0,...,0,1.0,7,7,0,1.000000,7,7,0,1.000000
3,bank_marketing,categorical,10,10,10,0,1.0,10,10,0,...,0,1.0,10,10,0,1.000000,10,10,0,1.000000
4,diabetes_130us,numeric,11,11,11,0,1.0,11,11,0,...,0,1.0,11,11,0,1.000000,11,11,0,1.000000
5,diabetes_130us,categorical,37,37,37,0,1.0,37,37,0,...,0,1.0,37,34,3,0.918919,37,34,3,0.918919


In [60]:
# =============================================================================
# SECTION 19 — REPETITION-LEVEL AGGREGATION
# =============================================================================

REPETITION_AGGREGATION_DF = DATASET_AGGREGATION_DF.copy()
REPETITION_AGGREGATION_DF.insert(1, "repetition", 1)
REPETITION_AGGREGATION_DF["repetition_source"] = (
    "Notebook 13 canonical final synthetic CSV; one observed generation"
)
REPETITION_AGGREGATION_DF["repetition_status"] = "OBSERVED"

REPETITION_AGGREGATION_DF.to_csv(
    NB14_AGG_ROOT / "repetition_level_fidelity_aggregation.csv",
    index=False
)

display(REPETITION_AGGREGATION_DF)
print(
    "Note: multi-repetition means/standard deviations require separate "
    "synthetic outputs generated from each repetition."
)

,dataset_id,repetition,feature_type,feature_count,real_rows,synthetic_rows,target_column,mean_mae_distribution_summary,median_mae_distribution_summary,valid_mae_distribution_summary_features,...,undefined_pearson_distribution_correlation_features,coverage_pearson_distribution_correlation,mean_spearman_distribution_correlation,median_spearman_distribution_correlation,valid_spearman_distribution_correlation_features,applicable_spearman_distribution_correlation_features,undefined_spearman_distribution_correlation_features,coverage_spearman_distribution_correlation,repetition_source,repetition_status
0,adult_income,1,numeric,6,7327,34189,income,3913.150495,172.560099,6,...,0,1.000000,0.797656,0.958160,6,6,0,1.000000,Notebook 13 canonical final synthetic CSV; one...,OBSERVED
1,adult_income,1,categorical,9,7327,34189,income,0.008035,0.003985,9,...,0,1.000000,0.908526,0.988235,9,9,0,1.000000,Notebook 13 canonical final synthetic CSV; one...,OBSERVED
2,bank_marketing,1,numeric,7,6782,31647,y,299.726011,3.702970,7,...,0,1.000000,0.899475,0.998630,7,7,0,1.000000,Notebook 13 canonical final synthetic CSV; one...,OBSERVED
3,bank_marketing,1,categorical,10,6782,31647,y,0.025484,0.009211,10,...,0,1.000000,0.980000,1.000000,10,10,0,1.000000,Notebook 13 canonical final synthetic CSV; one...,OBSERVED
4,diabetes_130us,1,numeric,11,15265,71236,readmitted,1.317957,0.861386,11,...,0,1.000000,0.859316,0.892363,11,11,0,1.000000,Notebook 13 canonical final synthetic CSV; one...,OBSERVED
5,diabetes_130us,1,categorical,37,15265,71236,readmitted,0.037963,0.002009,37,...,3,0.918919,0.745646,1.000000,34,37,3,0.918919,Notebook 13 canonical final synthetic CSV; one...,OBSERVED


Note: multi-repetition means/standard deviations require separate synthetic outputs generated from each repetition.


In [61]:
# =============================================================================
# SECTION 20 — MODEL-LEVEL AGGREGATION
# =============================================================================

MODEL_AGGREGATION_DF = DATASET_AGGREGATION_DF.copy()
MODEL_AGGREGATION_DF.insert(1, "model_id", "not_attributed")
MODEL_AGGREGATION_DF["model_checkpoint"] = pd.NA
MODEL_AGGREGATION_DF["model_attribution_status"] = (
    "UNAVAILABLE_FROM_CANONICAL_SYNTHETIC_CSV"
)

MODEL_AGGREGATION_DF.to_csv(
    NB14_AGG_ROOT / "model_level_fidelity_aggregation.csv",
    index=False
)

print("✓ Model-level aggregation artifact created.")
print(
    "Model-specific comparisons should be added only after Notebook 13 "
    "records the exact generating checkpoint for each synthetic output."
)

✓ Model-level aggregation artifact created.
Model-specific comparisons should be added only after Notebook 13 records the exact generating checkpoint for each synthetic output.


In [63]:
# =============================================================================
# SECTION 21 — SAVE FIDELITY RESULTS
# =============================================================================

print("=" * 100)
print("SECTION 21 — SAVE FIDELITY RESULTS")
print("=" * 100)

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. Validate Required Directories and Inputs
# -----------------------------------------------------------------------------

required_directories = [
    NB14_CONFIG_ROOT,
    NB14_INPUT_ROOT,
    NB14_FEATURE_ROOT,
    NB14_DATASET_ROOT,
    NB14_AGG_ROOT,
    NB14_VALIDATION_ROOT,
    NB14_LOG_ROOT,
]

for directory in required_directories:
    Path(directory).mkdir(parents=True, exist_ok=True)

required_variables = [
    "DATASET_IDS",
    "TARGET_COLUMNS",
    "REAL_TEST_PATHS",
    "SYNTHETIC_PATHS",
    "FEATURE_RESULTS_DF",
    "DATASET_AGGREGATION_DF",
    "REPETITION_AGGREGATION_DF",
    "MODEL_AGGREGATION_DF",
    "MASTER_SEED",
    "QUANTILE_GRID_SIZE",
    "HISTOGRAM_BINS",
    "PROBABILITY_SMOOTHING",
    "MAX_CATEGORY_LEVELS",
    "MAX_CORRELATION_FEATURES",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Required Notebook 14 variables are missing: "
        f"{missing_variables}"
    )

# -----------------------------------------------------------------------------
# 2. SHA-256 Fingerprinting Utility
# -----------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    """Return the SHA-256 checksum of a file using bounded memory."""
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(f"Cannot fingerprint missing file: {path}")

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


def artifact_record(name, path):
    """Create a reproducible metadata record for a saved artifact."""
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            f"Expected artifact does not exist: {path}"
        )

    return {
        "artifact_name": name,
        "path": str(path.resolve()),
        "exists": True,
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }


# -----------------------------------------------------------------------------
# 3. Register Principal Output Artifacts
# -----------------------------------------------------------------------------

OUTPUT_PATHS = {
    "schema_validation": (
        NB14_VALIDATION_ROOT / "schema_validation.csv"
    ),
    "feature_alignment": (
        NB14_VALIDATION_ROOT / "feature_alignment.csv"
    ),
    "feature_results": (
        NB14_FEATURE_ROOT / "feature_level_fidelity_results.csv"
    ),
    "dataset_aggregation": (
        NB14_DATASET_ROOT / "dataset_level_fidelity_aggregation.csv"
    ),
    "repetition_aggregation": (
        NB14_AGG_ROOT / "repetition_level_fidelity_aggregation.csv"
    ),
    "model_aggregation": (
        NB14_AGG_ROOT / "model_level_fidelity_aggregation.csv"
    ),
    "dependency_details": (
        NB14_FEATURE_ROOT / "dependency_preservation_details.csv"
    ),
    "dependency_summary": (
        NB14_DATASET_ROOT / "dependency_preservation_summary.csv"
    ),
    "feature_metric_coverage": (
        NB14_VALIDATION_ROOT / "feature_metric_coverage.csv"
    ),
    "dataset_metric_coverage": (
        NB14_VALIDATION_ROOT / "dataset_level_metric_coverage.csv"
    ),
    "repetition_metric_aggregation": (
        NB14_AGG_ROOT / "repetition_level_fidelity_aggregation.csv"
    ),
    "model_metric_aggregation": (
        NB14_AGG_ROOT / "model_level_fidelity_aggregation.csv"
    ),
}

# -----------------------------------------------------------------------------
# 4. Validate Principal Outputs Before Auditing
# -----------------------------------------------------------------------------

missing_outputs = [
    name for name, path in OUTPUT_PATHS.items()
    if not Path(path).is_file()
]

if missing_outputs:
    raise FileNotFoundError(
        "Cannot finalize Notebook 14 because these outputs are missing: "
        f"{missing_outputs}"
    )

if FEATURE_RESULTS_DF.empty:
    raise ValueError("FEATURE_RESULTS_DF is empty.")

if DATASET_AGGREGATION_DF.empty:
    raise ValueError("DATASET_AGGREGATION_DF is empty.")

if REPETITION_AGGREGATION_DF.empty:
    raise ValueError("REPETITION_AGGREGATION_DF is empty.")

if MODEL_AGGREGATION_DF.empty:
    raise ValueError("MODEL_AGGREGATION_DF is empty.")

# -----------------------------------------------------------------------------
# 5. Audit Real Test and Synthetic Input Files
# -----------------------------------------------------------------------------

INPUT_AUDIT_ROWS = []

for dataset_id in DATASET_IDS:
    dataset_paths = [
        ("real_test", REAL_TEST_PATHS[dataset_id]),
        ("synthetic", SYNTHETIC_PATHS[dataset_id]),
    ]

    for input_type, path in dataset_paths:
        path = Path(path)

        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {input_type} input for {dataset_id}: {path}"
            )

        INPUT_AUDIT_ROWS.append({
            "dataset_id": dataset_id,
            "input_type": input_type,
            "path": str(path.resolve()),
            "exists": True,
            "bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        })

INPUT_AUDIT_DF = pd.DataFrame(INPUT_AUDIT_ROWS)

INPUT_AUDIT_PATH = (
    NB14_INPUT_ROOT / "input_fingerprints.csv"
)

INPUT_AUDIT_DF.to_csv(
    INPUT_AUDIT_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 6. Build Configuration and Methodology Metadata
# -----------------------------------------------------------------------------

CONFIGURATION = {
    "notebook_name": NOTEBOOK_NAME,
    "notebook_version": NOTEBOOK_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook_02_root": str(NB02_ROOT),
    "notebook_13_root": str(NB13_ROOT),
    "notebook_14_root": str(NB14_ROOT),
    "datasets": list(DATASET_IDS),
    "target_columns": TARGET_COLUMNS,
    "real_reference": (
        "Notebook 02 held-out test split; no synthetic training on test data "
        "is assumed by this evaluation."
    ),
    "synthetic_reference": (
        "Notebook 13 canonical final synthetic CSV."
    ),
    "master_seed": int(MASTER_SEED),
    "quantile_grid_size": int(QUANTILE_GRID_SIZE),
    "histogram_bins": int(HISTOGRAM_BINS),
    "probability_smoothing": float(PROBABILITY_SMOOTHING),
    "max_category_levels": int(MAX_CATEGORY_LEVELS),
    "max_correlation_features": int(MAX_CORRELATION_FEATURES),
    "metric_interpretation": {
        "mae_rmse_r2": (
            "Descriptive comparisons between aligned distribution-summary "
            "vectors; not row-wise prediction metrics."
        ),
        "pearson_spearman": (
            "Correlation between aligned real and synthetic summary vectors; "
            "not a measure of multivariate dependency preservation."
        ),
        "wasserstein": (
            "Univariate numeric distribution distance in the original "
            "feature's units; lower is generally better."
        ),
        "js": (
            "Jensen-Shannon divergence over aligned histogram or category "
            "probability vectors; base 2."
        ),
        "kl": (
            "Directional Kullback-Leibler divergence with additive "
            "probability smoothing; lower is generally better."
        ),
        "dependency": (
            "Numeric pairwise Pearson/Spearman correlation differences and "
            "categorical-feature association with the target using "
            "Cramer's V."
        ),
    },
    "repetition_note": (
        "One canonical synthetic output per dataset is evaluated. "
        "This does not estimate between-repetition variability."
    ),
    "model_attribution_note": (
        "Model checkpoint attribution is not inferred from filenames or "
        "checkpoint availability. Notebook 13 must record the actual "
        "generating checkpoint."
    ),
    "data_handling_note": (
        "Synthetic domain violations and missing values are evaluated as "
        "observed; this section does not clip, impute, or silently repair "
        "synthetic data."
    ),
}

CONFIG_PATH = (
    NB14_CONFIG_ROOT / "statistical_fidelity_configuration.json"
)

with CONFIG_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        CONFIGURATION,
        handle,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

# -----------------------------------------------------------------------------
# 7. Reload and Validate Saved Artifacts
# -----------------------------------------------------------------------------

RELOAD_VALIDATION_ROWS = []

for name, path in OUTPUT_PATHS.items():
    path = Path(path)

    try:
        # Validate CSV readability without retaining a second full dataset.
        row_count = 0

        for chunk in pd.read_csv(
            path,
            chunksize=100_000,
            low_memory=False,
        ):
            row_count += len(chunk)

        RELOAD_VALIDATION_ROWS.append({
            "artifact_name": name,
            "path": str(path),
            "reload_status": "PASS",
            "rows_read": int(row_count),
            "error": None,
        })

    except Exception as exc:
        RELOAD_VALIDATION_ROWS.append({
            "artifact_name": name,
            "path": str(path),
            "reload_status": "FAIL",
            "rows_read": None,
            "error": str(exc),
        })

RELOAD_VALIDATION_DF = pd.DataFrame(RELOAD_VALIDATION_ROWS)

if not RELOAD_VALIDATION_DF["reload_status"].eq("PASS").all():
    failed = RELOAD_VALIDATION_DF.loc[
        RELOAD_VALIDATION_DF["reload_status"].ne("PASS")
    ]

    raise RuntimeError(
        "One or more saved CSV artifacts failed reload validation:\n"
        f"{failed.to_string(index=False)}"
    )

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    CONFIGURATION_RELOADED = json.load(handle)

if CONFIGURATION_RELOADED.get("datasets") != list(DATASET_IDS):
    raise ValueError(
        "Configuration reload validation failed: dataset registry mismatch."
    )

RELOAD_VALIDATION_PATH = (
    NB14_VALIDATION_ROOT / "artifact_reload_validation.csv"
)

RELOAD_VALIDATION_DF.to_csv(
    RELOAD_VALIDATION_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 8. Create Result Fingerprints
# -----------------------------------------------------------------------------

RESULT_FINGERPRINT_ROWS = [
    artifact_record(name, path)
    for name, path in OUTPUT_PATHS.items()
]

RESULT_FINGERPRINT_ROWS.append(
    artifact_record("configuration", CONFIG_PATH)
)

RESULT_FINGERPRINT_ROWS.append(
    artifact_record("input_audit", INPUT_AUDIT_PATH)
)

RESULT_FINGERPRINT_ROWS.append(
    artifact_record("reload_validation", RELOAD_VALIDATION_PATH)
)

RESULT_FINGERPRINT_DF = pd.DataFrame(
    RESULT_FINGERPRINT_ROWS
)

RESULT_FINGERPRINT_PATH = (
    NB14_INPUT_ROOT / "result_fingerprints.csv"
)

RESULT_FINGERPRINT_DF.to_csv(
    RESULT_FINGERPRINT_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 9. Final Integrity Checks
# -----------------------------------------------------------------------------

if RESULT_FINGERPRINT_DF["sha256"].isna().any():
    raise ValueError("Some output artifacts have missing SHA-256 fingerprints.")

if RESULT_FINGERPRINT_DF["sha256"].duplicated().any():
    print(
        "NOTE: Some artifacts have identical content hashes. "
        "This can be valid when artifacts contain identical data."
    )

print("=" * 100)
print("SECTION 21 — FINALIZATION SUMMARY")
print("=" * 100)
print(f"Datasets evaluated          : {len(DATASET_IDS)}")
print(f"Input files fingerprinted   : {len(INPUT_AUDIT_DF)}")
print(f"Principal artifacts checked : {len(OUTPUT_PATHS)}")
print(f"Artifacts reload-validated  : {len(RELOAD_VALIDATION_DF)}")
print(f"Reload validation failures  : {(RELOAD_VALIDATION_DF['reload_status'] != 'PASS').sum()}")
print(f"Configuration saved        : {CONFIG_PATH}")
print(f"Input audit saved           : {INPUT_AUDIT_PATH}")
print(f"Result audit saved          : {RESULT_FINGERPRINT_PATH}")
print(f"Reload validation saved     : {RELOAD_VALIDATION_PATH}")
print("STATUS: PASS")

SECTION 21 — SAVE FIDELITY RESULTS
NOTE: Some artifacts have identical content hashes. This can be valid when artifacts contain identical data.
SECTION 21 — FINALIZATION SUMMARY
Datasets evaluated          : 3
Input files fingerprinted   : 6
Principal artifacts checked : 12
Artifacts reload-validated  : 12
Reload validation failures  : 0
Configuration saved        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/configuration/statistical_fidelity_configuration.json
Input audit saved           : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/input_audit/input_fingerprints.csv
Result audit saved          : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/input_audit/result_fingerprints.csv
Reload validation saved     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/artifact_reload_validation.csv
STATUS: PASS


In [65]:
# =============================================================================
# SECTION 22 — VALIDATE COVERAGE AND ARTIFACT INTEGRITY
# =============================================================================

print("=" * 100)
print("SECTION 22 — VALIDATE COVERAGE AND ARTIFACT INTEGRITY")
print("=" * 100)

import json
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. Validate Required Inputs
# -----------------------------------------------------------------------------

required_variables = [
    "DATASET_IDS",
    "REAL_EVAL_DATA",
    "SYNTHETIC_EVAL_DATA",
    "FEATURE_RESULTS_DF",
    "TARGET_COLUMNS",
    "SCHEMA_VALIDATION_DF",
    "NB14_VALIDATION_ROOT",
    "OUTPUT_PATHS",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        f"Required variables are missing: {missing_variables}"
    )

NB14_VALIDATION_ROOT = Path(NB14_VALIDATION_ROOT)
NB14_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

required_result_columns = {
    "dataset_id",
    "feature",
    "feature_type",
    "mae_distribution_summary",
    "rmse_distribution_summary",
    "r2_distribution_summary",
    "wasserstein_distance",
    "js_divergence",
    "kl_real_to_synthetic",
    "kl_synthetic_to_real",
    "pearson_distribution_correlation",
    "spearman_distribution_correlation",
}

missing_result_columns = (
    required_result_columns - set(FEATURE_RESULTS_DF.columns)
)

if missing_result_columns:
    raise ValueError(
        "Feature-level results are missing required columns: "
        f"{sorted(missing_result_columns)}"
    )

if FEATURE_RESULTS_DF.empty:
    raise ValueError("FEATURE_RESULTS_DF is empty.")

if FEATURE_RESULTS_DF.duplicated(
    ["dataset_id", "feature"]
).any():
    duplicate_rows = FEATURE_RESULTS_DF.loc[
        FEATURE_RESULTS_DF.duplicated(
            ["dataset_id", "feature"], keep=False
        ),
        ["dataset_id", "feature"],
    ]

    raise ValueError(
        "Duplicate dataset-feature records detected:\n"
        f"{duplicate_rows.to_string(index=False)}"
    )

# -----------------------------------------------------------------------------
# 2. Define Metric Applicability
# -----------------------------------------------------------------------------

# Wasserstein distance is applicable to numeric features only.
# The other metrics are expected for numeric and categorical features.
METRIC_APPLICABILITY = {
    "mae_distribution_summary": {"numeric", "categorical"},
    "rmse_distribution_summary": {"numeric", "categorical"},
    "r2_distribution_summary": {"numeric", "categorical"},
    "wasserstein_distance": {"numeric"},
    "js_divergence": {"numeric", "categorical"},
    "kl_real_to_synthetic": {"numeric", "categorical"},
    "kl_synthetic_to_real": {"numeric", "categorical"},
    "pearson_distribution_correlation": {"numeric", "categorical"},
    "spearman_distribution_correlation": {"numeric", "categorical"},
}

REQUIRED_COVERAGE_METRICS = [
    "mae_distribution_summary",
    "rmse_distribution_summary",
    "js_divergence",
    "kl_real_to_synthetic",
    "kl_synthetic_to_real",
]

# -----------------------------------------------------------------------------
# 3. Validate Feature Coverage, Schema, and Metric Availability
# -----------------------------------------------------------------------------

COVERAGE_ROWS = []

EXPECTED_FEATURE_COUNTS = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 48,
}

schema_required_columns = {"dataset_id", "schema_status"}

if not schema_required_columns.issubset(
    SCHEMA_VALIDATION_DF.columns
):
    raise ValueError(
        "SCHEMA_VALIDATION_DF must contain dataset_id and schema_status."
    )

for dataset_id in DATASET_IDS:
    if dataset_id not in REAL_EVAL_DATA:
        raise KeyError(
            f"Real evaluation data missing for {dataset_id}."
        )

    if dataset_id not in SYNTHETIC_EVAL_DATA:
        raise KeyError(
            f"Synthetic evaluation data missing for {dataset_id}."
        )

    real_df = REAL_EVAL_DATA[dataset_id]
    synthetic_df = SYNTHETIC_EVAL_DATA[dataset_id]

    expected_features = set(real_df.columns)

    observed = FEATURE_RESULTS_DF.loc[
        FEATURE_RESULTS_DF["dataset_id"].eq(dataset_id)
    ].copy()

    observed_features = set(observed["feature"].astype(str))

    missing_features = sorted(
        expected_features - observed_features
    )
    unexpected_features = sorted(
        observed_features - expected_features
    )

    expected_count = EXPECTED_FEATURE_COUNTS.get(
        dataset_id, len(expected_features)
    )

    unique_feature_count = observed["feature"].nunique()

    feature_coverage_pass = (
        len(observed) == len(expected_features)
        and unique_feature_count == len(expected_features)
        and not missing_features
        and not unexpected_features
        and len(expected_features) == expected_count
    )

    target = TARGET_COLUMNS[dataset_id]
    target_present = target in expected_features
    target_in_results = target in observed_features

    schema_rows = SCHEMA_VALIDATION_DF.loc[
        SCHEMA_VALIDATION_DF["dataset_id"].eq(dataset_id),
        "schema_status",
    ]

    if len(schema_rows) != 1:
        schema_pass = False
        schema_status_detail = (
            f"Expected one schema status row; found {len(schema_rows)}"
        )
    else:
        schema_pass = str(schema_rows.iloc[0]).upper() == "PASS"
        schema_status_detail = str(schema_rows.iloc[0])

    row = {
        "dataset_id": dataset_id,
        "expected_feature_count": int(expected_count),
        "real_feature_count": int(len(expected_features)),
        "observed_feature_count": int(unique_feature_count),
        "observed_result_rows": int(len(observed)),
        "missing_features": json.dumps(missing_features),
        "unexpected_features": json.dumps(unexpected_features),
        "target_column": target,
        "target_present_in_real": bool(target_present),
        "target_present_in_results": bool(target_in_results),
        "feature_coverage_pass": bool(feature_coverage_pass),
        "real_rows": int(len(real_df)),
        "synthetic_rows": int(len(synthetic_df)),
        "schema_pass": bool(schema_pass),
        "schema_status_detail": schema_status_detail,
    }

    metric_failures = []

    for metric in REQUIRED_COVERAGE_METRICS:
        applicable_types = METRIC_APPLICABILITY[metric]

        applicable = observed.loc[
            observed["feature_type"].isin(applicable_types)
        ]

        metric_values = pd.to_numeric(
            applicable[metric],
            errors="coerce",
        ).replace([np.inf, -np.inf], np.nan)

        applicable_count = int(len(applicable))
        valid_count = int(metric_values.notna().sum())
        undefined_count = applicable_count - valid_count

        coverage_rate = (
            valid_count / applicable_count
            if applicable_count
            else np.nan
        )

        row[f"applicable_{metric}_count"] = applicable_count
        row[f"valid_{metric}_count"] = valid_count
        row[f"undefined_{metric}_count"] = undefined_count
        row[f"coverage_{metric}"] = coverage_rate

        if applicable_count == 0 or valid_count != applicable_count:
            metric_failures.append(metric)

    # Record coverage for every supported metric, including Wasserstein.
    for metric, applicable_types in METRIC_APPLICABILITY.items():
        if metric in REQUIRED_COVERAGE_METRICS:
            continue

        applicable = observed.loc[
            observed["feature_type"].isin(applicable_types)
        ]

        metric_values = pd.to_numeric(
            applicable[metric],
            errors="coerce",
        ).replace([np.inf, -np.inf], np.nan)

        applicable_count = int(len(applicable))
        valid_count = int(metric_values.notna().sum())
        undefined_count = applicable_count - valid_count

        row[f"applicable_{metric}_count"] = applicable_count
        row[f"valid_{metric}_count"] = valid_count
        row[f"undefined_{metric}_count"] = undefined_count
        row[f"coverage_{metric}"] = (
            valid_count / applicable_count
            if applicable_count
            else np.nan
        )

    row["missing_required_metrics"] = json.dumps(metric_failures)

    overall_pass = (
        feature_coverage_pass
        and target_present
        and target_in_results
        and schema_pass
        and not metric_failures
    )

    row["coverage_status"] = "PASS" if overall_pass else "FAIL"

    COVERAGE_ROWS.append(row)

COVERAGE_DF = pd.DataFrame(COVERAGE_ROWS)

COVERAGE_PATH = (
    NB14_VALIDATION_ROOT / "fidelity_coverage_validation.csv"
)

COVERAGE_DF.to_csv(
    COVERAGE_PATH,
    index=False,
    float_format="%.10g",
)

# -----------------------------------------------------------------------------
# 4. Validate Metric Ranges and Divergence Constraints
# -----------------------------------------------------------------------------

RANGE_CHECK_ROWS = []

for metric in (
    "js_divergence",
    "kl_real_to_synthetic",
    "kl_synthetic_to_real",
):
    values = pd.to_numeric(
        FEATURE_RESULTS_DF[metric],
        errors="coerce",
    ).replace([np.inf, -np.inf], np.nan)

    finite_values = values.dropna()

    negative_count = int((finite_values < -1e-10).sum())

    if metric == "js_divergence":
        out_of_range_count = int(
            (
                (finite_values < -1e-10)
                | (finite_values > 1.0 + 1e-8)
            ).sum()
        )
        expected_range = "[0, 1] for base-2 JS divergence"
    else:
        out_of_range_count = negative_count
        expected_range = "Nonnegative"

    RANGE_CHECK_ROWS.append({
        "metric": metric,
        "finite_value_count": int(len(finite_values)),
        "undefined_value_count": int(values.isna().sum()),
        "negative_value_count": negative_count,
        "out_of_range_count": out_of_range_count,
        "expected_range": expected_range,
        "status": (
            "PASS" if out_of_range_count == 0 else "FAIL"
        ),
    })

METRIC_RANGE_VALIDATION_DF = pd.DataFrame(RANGE_CHECK_ROWS)

METRIC_RANGE_PATH = (
    NB14_VALIDATION_ROOT / "metric_range_validation.csv"
)

METRIC_RANGE_VALIDATION_DF.to_csv(
    METRIC_RANGE_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 5. Validate Principal Saved Artifacts
# -----------------------------------------------------------------------------

ARTIFACT_CHECK_ROWS = []

for name, path in OUTPUT_PATHS.items():
    path = Path(path)
    exists = path.is_file()

    readable = False
    row_count = np.nan
    error_message = None

    try:
        if not exists:
            raise FileNotFoundError(
                f"Artifact does not exist: {path}"
            )

        if path.suffix.lower() == ".csv":
            row_count = 0

            for chunk in pd.read_csv(
                path,
                chunksize=100_000,
                low_memory=False,
            ):
                row_count += len(chunk)

        elif path.suffix.lower() == ".json":
            with path.open("r", encoding="utf-8") as handle:
                json.load(handle)

        readable = True

    except Exception as exc:
        error_message = str(exc)

    ARTIFACT_CHECK_ROWS.append({
        "artifact_name": name,
        "path": str(path),
        "exists": bool(exists),
        "readable": bool(readable),
        "rows_read": row_count,
        "status": "PASS" if exists and readable else "FAIL",
        "error": error_message,
    })

ARTIFACT_CHECK_DF = pd.DataFrame(ARTIFACT_CHECK_ROWS)

ARTIFACT_CHECK_PATH = (
    NB14_VALIDATION_ROOT / "artifact_readability_validation.csv"
)

ARTIFACT_CHECK_DF.to_csv(
    ARTIFACT_CHECK_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 6. Save Final Validation Summary
# -----------------------------------------------------------------------------

validation_summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "datasets_evaluated": int(len(DATASET_IDS)),
    "feature_coverage_pass_count": int(
        COVERAGE_DF["feature_coverage_pass"].sum()
    ),
    "schema_pass_count": int(COVERAGE_DF["schema_pass"].sum()),
    "coverage_pass_count": int(
        COVERAGE_DF["coverage_status"].eq("PASS").sum()
    ),
    "coverage_fail_count": int(
        COVERAGE_DF["coverage_status"].eq("FAIL").sum()
    ),
    "artifact_count": int(len(ARTIFACT_CHECK_DF)),
    "artifact_pass_count": int(
        ARTIFACT_CHECK_DF["status"].eq("PASS").sum()
    ),
    "artifact_fail_count": int(
        ARTIFACT_CHECK_DF["status"].eq("FAIL").sum()
    ),
    "metric_range_fail_count": int(
        METRIC_RANGE_VALIDATION_DF["status"].eq("FAIL").sum()
    ),
    "overall_status": (
        "PASS"
        if (
            COVERAGE_DF["coverage_status"].eq("PASS").all()
            and ARTIFACT_CHECK_DF["status"].eq("PASS").all()
            and METRIC_RANGE_VALIDATION_DF["status"].eq("PASS").all()
        )
        else "FAIL"
    ),
}

VALIDATION_SUMMARY_PATH = (
    NB14_VALIDATION_ROOT / "fidelity_validation_summary.json"
)

with VALIDATION_SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        validation_summary,
        handle,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

# -----------------------------------------------------------------------------
# 7. Display Results and Enforce Validation
# -----------------------------------------------------------------------------

print("\nFEATURE AND METRIC COVERAGE")
display(COVERAGE_DF)

print("\nMETRIC RANGE VALIDATION")
display(METRIC_RANGE_VALIDATION_DF)

print("\nARTIFACT READABILITY VALIDATION")
display(ARTIFACT_CHECK_DF)

print("\nFINAL VALIDATION SUMMARY")
print(json.dumps(validation_summary, indent=2))

print(f"\nCoverage report : {COVERAGE_PATH}")
print(f"Range report    : {METRIC_RANGE_PATH}")
print(f"Artifact report : {ARTIFACT_CHECK_PATH}")
print(f"Summary         : {VALIDATION_SUMMARY_PATH}")

if validation_summary["overall_status"] != "PASS":
    raise RuntimeError(
        "Notebook 14 validation did not pass. "
        "Review the coverage, metric range, and artifact reports."
    )

print("\nSTATUS: PASS")

SECTION 22 — VALIDATE COVERAGE AND ARTIFACT INTEGRITY

FEATURE AND METRIC COVERAGE


,dataset_id,expected_feature_count,real_feature_count,observed_feature_count,observed_result_rows,missing_features,unexpected_features,target_column,target_present_in_real,target_present_in_results,...,applicable_pearson_distribution_correlation_count,valid_pearson_distribution_correlation_count,undefined_pearson_distribution_correlation_count,coverage_pearson_distribution_correlation,applicable_spearman_distribution_correlation_count,valid_spearman_distribution_correlation_count,undefined_spearman_distribution_correlation_count,coverage_spearman_distribution_correlation,missing_required_metrics,coverage_status
0,adult_income,15,15,15,15,[],[],income,True,True,...,15,15,0,1.0000,15,15,0,1.0000,[],PASS
1,bank_marketing,17,17,17,17,[],[],y,True,True,...,17,17,0,1.0000,17,17,0,1.0000,[],PASS
2,diabetes_130us,48,48,48,48,[],[],readmitted,True,True,...,48,45,3,0.9375,48,45,3,0.9375,[],PASS



METRIC RANGE VALIDATION


,metric,finite_value_count,undefined_value_count,negative_value_count,out_of_range_count,expected_range,status
0,js_divergence,80,0,0,0,"[0, 1] for base-2 JS divergence",PASS
1,kl_real_to_synthetic,80,0,0,0,Nonnegative,PASS
2,kl_synthetic_to_real,80,0,0,0,Nonnegative,PASS



ARTIFACT READABILITY VALIDATION


,artifact_name,path,exists,readable,rows_read,status,error
0,schema_validation,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,3,PASS,None
1,feature_alignment,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,80,PASS,None
2,feature_results,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,80,PASS,None
3,dataset_aggregation,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,6,PASS,None
4,repetition_aggregation,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,6,PASS,None
5,model_aggregation,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,6,PASS,None
6,dependency_details,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,235,PASS,None
7,dependency_summary,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,9,PASS,None
8,feature_metric_coverage,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,27,PASS,None
9,dataset_metric_coverage,/content/drive/MyDrive/SPP_GAN_Research/result...,True,True,6,PASS,None



FINAL VALIDATION SUMMARY
{
  "created_utc": "2026-10-14T11:03:08.328822+00:00",
  "datasets_evaluated": 3,
  "feature_coverage_pass_count": 3,
  "schema_pass_count": 3,
  "coverage_pass_count": 3,
  "coverage_fail_count": 0,
  "artifact_count": 12,
  "artifact_pass_count": 12,
  "artifact_fail_count": 0,
  "metric_range_fail_count": 0,
  "overall_status": "PASS"
}

Coverage report : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/fidelity_coverage_validation.csv
Range report    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/metric_range_validation.csv
Artifact report : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/artifact_readability_validation.csv
Summary         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/validation/fidelity_validation_summary.json

STATUS: PASS


In [66]:
# =============================================================================
# SECTION 23 — COMPLETION SUMMARY
# =============================================================================

coverage_pass = bool(
    COVERAGE_DF["coverage_status"].eq("PASS").all()
)
schema_pass = bool(
    SCHEMA_VALIDATION_DF["schema_status"].eq("PASS").all()
)
artifacts_pass = bool(
    ARTIFACT_CHECK_DF["status"].eq("PASS").all()
)
input_files_pass = bool(
    INPUT_AUDIT_DF["exists"].all()
    and INPUT_AUDIT_DF["bytes"].gt(0).all()
)

feature_count = int(len(FEATURE_RESULTS_DF))
dataset_count = int(FEATURE_RESULTS_DF["dataset_id"].nunique())
numeric_feature_count = int(
    FEATURE_RESULTS_DF["feature_type"].eq("numeric").sum()
)
categorical_feature_count = int(
    FEATURE_RESULTS_DF["feature_type"].eq("categorical").sum()
)

completion_status = (
    "COMPLETED"
    if coverage_pass and schema_pass and artifacts_pass and input_files_pass
    else "REVIEW_REQUIRED"
)

completion_summary = {
    "notebook": NOTEBOOK_NAME,
    "version": NOTEBOOK_VERSION,
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "status": completion_status,
    "structural_validation": (
        "PASS"
        if coverage_pass and schema_pass and artifacts_pass and input_files_pass
        else "REVIEW_REQUIRED"
    ),
    "quality_interpretation": (
        "METRICS_REPORTED_REQUIRES_RESEARCH_INTERPRETATION"
    ),
    "datasets_expected": len(DATASET_IDS),
    "datasets_evaluated": dataset_count,
    "feature_result_rows": feature_count,
    "numeric_feature_rows": numeric_feature_count,
    "categorical_feature_rows": categorical_feature_count,
    "schema_validation_pass": schema_pass,
    "feature_coverage_pass": coverage_pass,
    "result_artifact_readability_pass": artifacts_pass,
    "input_fingerprint_validation_pass": input_files_pass,
    "real_reference": "Notebook 02 held-out test data",
    "synthetic_reference": "Notebook 13 final synthetic CSVs",
    "metrics": [
        "distribution-summary MAE",
        "distribution-summary RMSE",
        "distribution-summary R2",
        "Wasserstein distance",
        "Jensen-Shannon divergence",
        "KL divergence (real to synthetic)",
        "KL divergence (synthetic to real)",
        "Pearson correlation of distribution summaries",
        "Spearman correlation of distribution summaries",
        "numeric dependency preservation",
        "categorical-to-target dependency preservation",
    ],
    "repetition_aggregation": (
        "One observed synthetic generation per dataset; no independent "
        "repetition variance estimated."
    ),
    "model_aggregation": (
        "Checkpoint attribution unavailable from the canonical CSV alone."
    ),
    "important_limitations": [
        "Distribution-summary MAE/RMSE/R2 are not row-level prediction errors.",
        "Wasserstein distance is reported in original numeric feature units.",
        "KL divergence uses PROBABILITY_SMOOTHING.",
        "Categorical-to-target association does not establish causality.",
        "Statistical fidelity alone does not certify privacy or downstream utility.",
    ],
    "output_directories": {
        "configuration": str(NB14_CONFIG_ROOT),
        "input_audit": str(NB14_INPUT_ROOT),
        "feature_results": str(NB14_FEATURE_ROOT),
        "dataset_results": str(NB14_DATASET_ROOT),
        "aggregations": str(NB14_AGG_ROOT),
        "validation": str(NB14_VALIDATION_ROOT),
        "logs": str(NB14_LOG_ROOT),
    },
}

COMPLETION_SUMMARY_PATH = (
    NB14_ROOT / "notebook_14_completion_summary.json"
)
with open(COMPLETION_SUMMARY_PATH, "w", encoding="utf-8") as handle:
    json.dump(completion_summary, handle, indent=2, ensure_ascii=False)

# Verify the completion summary can be read back.
with open(COMPLETION_SUMMARY_PATH, "r", encoding="utf-8") as handle:
    completion_summary_check = json.load(handle)

print("=" * 100)
print("NOTEBOOK 14 — COMPLETION SUMMARY")
print("=" * 100)
print("Status                  :", completion_summary_check["status"])
print("Structural validation   :", completion_summary_check["structural_validation"])
print("Datasets evaluated      :", dataset_count, "/", len(DATASET_IDS))
print("Feature-level rows      :", feature_count)
print("Numeric feature rows    :", numeric_feature_count)
print("Categorical feature rows:", categorical_feature_count)
print("Schema validation       :", "PASS" if schema_pass else "REVIEW")
print("Feature coverage        :", "PASS" if coverage_pass else "REVIEW")
print("Artifact readability    :", "PASS" if artifacts_pass else "REVIEW")
print("Input fingerprints      :", "PASS" if input_files_pass else "REVIEW")
print("Completion summary      :", COMPLETION_SUMMARY_PATH)
print("Configuration           :", CONFIG_PATH)
print("Feature results         :", NB14_FEATURE_ROOT)
print("Dataset results         :", NB14_DATASET_ROOT)
print("Aggregations            :", NB14_AGG_ROOT)
print("Validation reports      :", NB14_VALIDATION_ROOT)
print("=" * 100)

NOTEBOOK 14 — COMPLETION SUMMARY
Status                  : COMPLETED
Structural validation   : PASS
Datasets evaluated      : 3 / 3
Feature-level rows      : 80
Numeric feature rows    : 24
Categorical feature rows: 56
Schema validation       : PASS
Feature coverage        : PASS
Artifact readability    : PASS
Input fingerprints      : PASS
Completion summary      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/notebook_14_completion_summary.json
Configuration           : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/configuration/statistical_fidelity_configuration.json
Feature results         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/feature_results
Dataset results         : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/dataset_results
Aggregations            : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_14/aggregations
Validation reports      : /content/drive/M